# ISM 6564 · Week 7 · Assignment 7 — Ship Northwind's Help-Centre Assistant

**ISM 6564 — Text Analytics · Fall 2026 · Week 7: Prompt Augmentation, RAG & Context Engineering**

A language model knows nothing about Northwind: not its return window, not what form LP-14 is for, not
where your parcel is. Everything it gets right about this company, it gets right because the application
put the fact into the prompt first. That is **prompt augmentation**, and this assignment is a team
building something that cannot work without it.

> **What you build:** a support assistant for Northwind's customers that answers from **two** sources it
> was never trained on. The **help centre** (34 articles, one of them an old policy nobody deleted, three
> of them internal staff pages) is searched: you cut it into passages two ways, one of them a **semantic
> cut** you design, and you measure which cut finds more. The customer's **own orders** are looked up:
> one fixed SQL query against Northwind's order database (SQLite, provided, read-only), scoped to the
> customer who is signed in. Passages and row go into the same prompt, so the assistant can say what the
> return policy is **and** where order NW-40402 has got to, and it must not say a word about anybody
> else's order.

> **How you get there, as a team:** write the pieces as functions, measure each one (the cut, the search,
> the guardrails, the look-up), write ten test questions of your own, design and measure one improvement,
> test the result **once** on 12 held-out questions nobody tuned for, and ship it as a **chat page** that
> runs on your laptop with Ollama. Do it together, in a shared GitHub repository, and finish with a
> release note to Northwind's head of support.

**Everything you need is in this notebook:** the deadlines, how your team works and submits, how it is
marked, the setup notes and every task. The lecture notebook (`ism6564-week07-lecture-notebook.ipynb`, on
Canvas) is your reference, and each task says which of its sections to read: §2.1 and §2.6 for the cuts,
§5.4 and §5.5 for memory and the order look-up. Reuse its code freely; every choice you make here has to
be yours and measured.


## When and how

- **Opens:** Wednesday, October 7, 2026 at 12:30 PM ET (start of Week 7 class).
- **Due:** Tuesday, October 13, 2026 at 11:59 PM ET (the evening before Week 8's class).
- **Late work:** accepted up to 24 hours late with a flat deduction of **20% of the assignment's points
  (20 points)**, applied automatically by Canvas. After Wednesday, October 14, 2026 at 11:59 PM ET it is not
  accepted and scores zero. The rule applies to your team's Canvas submission.
- **This assignment counts toward your assignment grade** (30% of the course total, spread over ten
  assignments, lowest dropped).

## A team assignment, built in a shared GitHub repository

You do this assignment **in your final-project team**. From this assignment on, your team works in, and
submits, **a GitHub repository**.

**It is bigger than the earlier assignments, on purpose.** Assignments 1–6 were sized for one person. This
one is sized for a team: two cutters, a search, an answerer with its guardrails, a database look-up, ten
test questions to write, an improvement to design and measure, a chat page to ship and try out, and a
release note. It is meant to be divided. A team that
leaves it to one member will run out of week, and the repository will show it.

**How a team assignment works**, on this and every remaining assignment:

- **One team, one submission.** One member submits on Canvas for the whole team. If a second member
  submits, that submission simply replaces the first, so agree who submits and what goes in.
- **Work together, divide the work, and plan the week.** Learning to do that is part of the assignment:
  split the tasks, set dates inside the week, and use each other's time well.
- **If you do more, bring your teammates along.** A member who ends up doing more of the work is expected
  to make the effort to help the others understand it: walk them through it and have them review it.
  Any member may be asked to explain any part.
- **A contribution statement goes in with the submission.** It says what each member contributed, and
  every member agrees to it before the team submits (`TEAM.md`, below).
- **A peer evaluation follows.** Every member is asked, privately, about each teammate's contribution
  (*Your share of the team's mark*, below).

### The repository

- **One private repository per team**, on GitHub, named `ism6564-a7-<your-team-name>`. Add every team
  member as a collaborator. Keep it private: sharing work with another team is an academic-integrity matter.
- **You must invite me to the repository.** Because it is private, I cannot see it, or mark it, until you
  do. On GitHub: *Settings → Collaborators → Add people*, and add **`prof-tcsmith`**
  (https://github.com/prof-tcsmith). Send the invitation when you create the repository, in the first
  day or two, not at the deadline.
- **What it holds:**
  - this notebook, run top to bottom with every output visible;
  - `ism6564-week07-app.py`, the chat page the notebook writes (Task 7);
  - your screenshot of the page (Task 7);
  - `TEAM.md`, your plan and your record (below);
  - `ai-usage/`, with one AI Usage Report per member, named `ai-usage/<Firstname-Lastname>.md` and
    **committed by the member it belongs to**.
- **`TEAM.md`.** *Before you build:* who owns which task, who reviews it, and when each piece is due
  inside the week. Commit this plan first. *When you finish:* the team's **contribution statement**,
  under that heading: two or three sentences per member on what they contributed (which tasks they
  built, what they reviewed), agreed by every member, and anything that did not go to plan.
- **Work in the repository all week.** Commit small and often, **from your own GitHub account**, with
  messages that say what changed. Give each task a branch and a pull request, and have another member
  review it before it is merged (the steps are below). Task 1 (the two cuts), Task 4 (memory and the order
  look-up) and Task 5a (your test questions) can be written in parallel from day one: Task 1's checks
  need only the small search model, and Task 4's first checks and Task 5a's check of your questions
  need no model and no other task. (The scripted conversation and the order questions in Task 4, and
  the scoring in Task 5a, wait until Tasks 1–3 are done and Ollama is running.)
- **A practical warning about notebooks.** Two people editing one notebook at the same time produces merge
  conflicts. Agree who has the notebook, pull before you start, and clear every output before you commit
  work in progress (**Clear All Outputs**, in the notebook's toolbar); run the whole notebook once, at the
  end, for the commit you submit.

### Working in the repository, step by step

Plain `git` in a terminal and github.com in a browser are enough. An editor's Git panel does the same
things; the steps do not change.

1. **Create the repository (one member).** On github.com, create a new repository named
   `ism6564-a7-<your-team-name>` and set it to **Private**.
2. **Add everyone.** In the repository, *Settings → Collaborators → Add people*: each teammate's GitHub
   username, and **`prof-tcsmith`**. Each person accepts the invitation GitHub emails them.
3. **Clone it (every member).** `git clone https://github.com/<owner>/ism6564-a7-<your-team-name>.git`,
   then `cd` into the folder, and put this notebook in it. The first time git talks to GitHub it asks you
   to sign in. If it asks for a password, it wants a *personal access token*, not your GitHub password
   (github.com: *Settings → Developer settings → Personal access tokens*).
4. **Make your commits yours.** GitHub credits a commit to the account whose email address is on it.
   Once, in the cloned folder: `git config user.name "Your Name"` and
   `git config user.email "the-address-on-your-GitHub-account"`.
5. **Commit the plan first.** Write `TEAM.md`, then `git add TEAM.md`,
   `git commit -m "Plan: who owns which task"` and `git push`.
6. **A branch per task.** `git switch main` and `git pull` to start from the latest version, then
   `git switch -c task2-search` (any name that says what the branch is for).
7. **Commit as you go.** `git add <the files you changed>`, then
   `git commit -m "Task 2: rrf and a fused, re-ranked search"`. Small commits, on several days.
8. **Push the branch.** `git push -u origin task2-search` the first time, plain `git push` after that.
9. **Open a pull request.** On github.com, open *Pull requests → New pull request*, choose your branch,
   and say what changed and what the reviewer should look at.
10. **A teammate reviews it.** On the pull request's *Files changed* tab they read the change, comment on
    it, and approve it or ask for changes. Fix what they found on the same branch and push again.
11. **Merge.** When it is approved, merge the pull request on github.com. Everyone then runs
    `git switch main` and `git pull` before starting the next piece.

### How your teamwork is marked (15 points)

I read the repository's **history**, not only its final state:

| What I look for | Points |
| --- | ---: |
| **Everyone built something.** Every member has commits of substance, from their own account, on the part `TEAM.md` gave them. | 5 |
| **The work was divided and built up over the week.** The plan in `TEAM.md` was committed before the code, the history matches the plan, and the commits fall on several different days. | 5 |
| **You reviewed each other's work.** Pull requests (or issues) with comments from a member who did not write the change, and commit messages that say what changed and why. | 5 |

**In Assignment 7 this section is marked gently.** This is the first assignment you do this way, and I
expect you to experiment and to learn how to work as a team in a repository. On the later assignments,
and on the final project, I expect to mark teamwork with the rubric above, as written.

Under that rubric, a repository with one or two large commits near the deadline — the work done somewhere
else and then posted — earns none of these 15 points, however good the notebook is. Any member may be
asked to explain any part of the notebook or the app, not only the part they wrote.

### Your share of the team's mark

The notebook, the app and the repository earn a **team mark**. **Your mark is your share of it**, on this
and every remaining assignment, and on the final project.

- **The peer evaluation sets your share.** By the assignment deadline, every member emails me, privately
  (subject `ISM 6564 — <team name> — peer evaluation, Assignment 7`), a rating **out of 10** for each
  teammate's contribution to this assignment, not including themselves, with a sentence of reasons for
  each. Ten means the teammate did their full share. Your share is the average of the ratings your
  teammates give you: **if your peers give you 8 out of 10, you receive 80% of the team's mark.**
- **The repository checks it.** I read the history — your commits, your pull requests, your reviews of
  other members' work — as a check on the ratings. Where it clearly contradicts them (a member rated 10
  who built nothing, a member rated low who built a large part), I adjust the share, in either direction.
- **If you do not send your peer evaluation by the deadline, your own share is at most 60%: the lower of
  60% and what your teammates rated you.** Rating your teammates is part of the work.
- A team where everyone did their share, and the repository shows it, gives everyone 10.

### What to submit

**One member submits for the team** on Canvas, before the deadline, as text (Assignment 7 is a Canvas group
assignment, so one submission counts for every member of your team):

1. the **repository URL**,
2. the **full commit hash** of the version to be marked (`git rev-parse HEAD`, or copy it from the commit's
   page on GitHub), and
3. the team's **contribution statement**, copied from `TEAM.md`: what each member contributed.

There is one submission per team. If a second member submits, it replaces the first.

**Every member** also emails me their peer evaluation by the deadline (*Your share of the team's mark*,
above).

Only what is in the repository **at that commit** is marked, and nothing is uploaded to Canvas. I can
only mark a repository I have been invited to (`prof-tcsmith`, above). The commit
must contain everything listed under *The repository*, including **every member's own** AI Usage Report
(next).

### Your AI Usage Report: a short reflection, not a transcript

AI use is expected on every assignment, so the report is never optional. From this assignment on it is a
**short reflection on how you used AI in your part of the work**. You do **not** need to collect or paste
your AI conversations; this replaces the full-transcript report of Assignments 1 to 6. Each member writes
their own, in their own words, as `ai-usage/<Firstname-Lastname>.md`. Half a page is enough:

- **What you used.** The tools and models, and what you used each one for. Count everything: writing and
  debugging code, understanding the lecture, planning and scheduling the team's week, drafting a message
  to a teammate, reviewing a pull request.
- **Two or three specific examples.** For each one: what you asked for, what came back, and what you did
  with it. Include one where the AI was wrong or unhelpful, and how you noticed.
- **How you checked** that what you used was correct before it went into the repository.
- **What you would do differently next time.**

**An example**, about the right length:

> **Tools.** Claude (Sonnet 5.5) in VS Code for code; ChatGPT for planning and for one message to the team.
>
> **How I used them.** My part was Task 4, the order look-up. I asked Claude to explain what a
> parameterised query is and why the lecture passes the values separately. Then I wrote `lookup_order`
> myself and asked it to review my query. It pointed out that the query was not limited to the signed-in
> customer, so another customer's order came back; I added `AND customer_id = ?` and the notebook's check
> passed. I also asked it to write `order_facts`. Its version printed `None` for an order with no tracking
> number, which the task says not to do, so I rewrote that part by hand.
>
> Outside the code, I asked ChatGPT to suggest an order for our tasks across the week. We moved Task 6
> after Task 5 and wrote the plan in `TEAM.md` ourselves. I also used it to shorten a message asking a
> teammate to review my pull request.
>
> **How I checked.** I ran the Task 4 checks after every change and tried two order numbers by hand: one
> of Dana's and one from another account.
>
> **Next time.** I would ask for an explanation before asking for code. The explanation is what let me
> see the missing customer check.

The report is a gate on your **own** mark (see the rubric).

## Team

*Team name, repository URL, and members (full names and GitHub usernames):* ___

**Where AI may not write the text.** The answer cells in Tasks 1 to 7, the Task 8 release note and
`TEAM.md` must be written by your team. You may ask an AI assistant to critique a draft you wrote, not to
write or rewrite it; AI-polished text there counts as AI-written. Everywhere else, including the code, AI
help is allowed, and your AI Usage Report says how you used it.


## How it is marked

| Component | Points |
| --- | ---: |
| **Task 1 — the index, cut two ways.** `chunk`, the section cut, passes its checks (2); `chunk_semantic`, your semantic cut, passes its checks (4); both cuts scored and one chosen (1). Written: the numbers for both and the reason for your choice (3). | 10 |
| **Task 2 — the search.** `rrf` and `search` pass the self-checks (3); scored on its own, by kind of question, with your k (2). Written: the search you chose and why, with numbers (3). | 8 |
| **Task 3 — the answerer and its guardrails.** The rules and the augmented prompt pass the self-check (3); a hard filter that keeps the old and internal pages out (3); a decline rule and an answer-check decision, measured against at least two alternatives (5). Written: the trade you chose, with numbers (6). | 17 |
| **Task 4 — memory and the order database.** `remember` and `lookup_order` pass their checks: one parameterised query, a join, scoped to the customer (5); `order_facts` and `needs_person` pass theirs, and the scripted conversation passes (4); the order questions scored with pages only and with the row (2). Written: what the row changed, and an answer you checked against its key (2). | 13 |
| **Task 5 — extend it.** Ten golden questions of your own that pass the checks (5); one improvement aimed at a failure you saw, built and measured (4). Written: what the questions cover, what the improvement changed, keep it or not (3). | 12 |
| **Task 6 — measure what you ship.** Your final design against the baseline on the tuning questions, your own and the order questions (3); the 12 held-out questions, run once (2). Written: what held up and what did not (3). | 8 |
| **Task 7 — ship it.** The app file written, imported, and answering exactly as the notebook (4); the page running, screenshot in the repository (3); five conversations you wrote, including an attack and an order of the signed-in customer's, in the table, with the written answer (3). | 10 |
| **Task 8 — the release note.** A committed go / no-go with at least four measured numbers, one from the held-out questions (3); what is filtered, looked up, declined and handed to a person (2); what you would watch after launch (2). | 7 |
| **Teamwork in the repository.** Everyone built something (5); divided and built up over the week (5); you reviewed each other's work (5). Marked gently in this first team assignment; see above. | 15 |
| **AI Usage Report** (each member's own short reflection, with specific examples; no transcript needed) | 0 (gate, on your own mark: −5 to −10 if yours is missing or gives no specific examples) |
| **Total** | **100** |

These 100 points are the **team's** mark. Each member receives their **share** of it, set by the team's peer
evaluations and checked against the repository's history (*Your share of the team's mark*, above); the AI
Usage Report gate is then applied to each member's own mark.

**Reproducibility is part of every task:** seeds untouched, the notebook runs top to bottom, outputs
visible, no API key anywhere, no stray scratch cells. **Do not tune on the held-out questions**: Task 6
runs them once, at the end. A design that was changed after looking at held-out answers is marked as not
held out.


## How to work through this notebook

- Every place you write something is marked `# TODO`. Everything else — the data, the models, the
  scorer, the answer pipeline, the app writer — is written for you and mirrors the lecture notebook.
- **The notebook runs top to bottom right now, with nothing filled in.** Cells that depend on an
  unfinished `# TODO` print `SKIPPED` and move on. Fill the tasks in order and re-run from the top.
  (Two parts can be written early, because their checks need nothing else: the four functions of Task 4,
  and your ten questions in Task 5a.)
- Markdown cells headed **Your answer** are marked. Replace the placeholder text.
- Do not change `SEED`, `MODEL` or the decoding settings in `ask_messages`.
- **Divide it, then read each other's.** A split that works for four or five: one member on Task 1 (the
  two cuts), one on Tasks 2 and 3 (the search, the answerer and its guardrails), one on Task 4 (the
  order database) and the chat page, one on Task 5a (the test questions), and everyone on Task 5b's
  improvement and the release note. Every member must understand the whole notebook.

### Before you start

- **Run the lecture notebook** once, and do the **practice notebook** (Canvas): it rehearses the core
  mechanics on a smaller help centre. Read lecture notebook §2.1 and §2.6 (the four ways to cut a page,
  and the cuts measured) before Task 1, and §5.5 (a table lookup in SQLite) before Task 4.
- **Where to run it: a team member's own computer, with Ollama** — the Week 6 setup (setup guide §5,
  Canvas). Pull the model once: `ollama pull qwen2.5:1.5b` (about 1 GB). Ollama writes the answers; the two
  small search models (about 90 MB each) run in the notebook, on a GPU if you have one and otherwise on
  the CPU. Colab cannot run Ollama or the chat page, so run this notebook locally. More than one member
  should have it running: the work divides only if more than one of you can run it.
- **Test your install** with the Week 5 setup notebook (`ism6564-local-llm-setup.ipynb`, Canvas). Your
  machine is your responsibility; there is no tech support for personal installs.
- **Packages:** `sentence-transformers`, `rank-bm25` and `openai` (it talks to Ollama) are in the course
  environment; if one is missing, the first cell installs it where it can and prints the command to run
  where it cannot. Task 7 also needs **Streamlit**, as in Week 6,
  and the course environment does not list it. In a terminal where the course environment is active (the
  environment this notebook's kernel uses), run `uv pip install streamlit`. Run it again if you re-sync
  the course environment: `uv sync` removes packages the environment does not list. (The course
  environment has no `pip` of its own, so a bare `pip install` either fails or puts Streamlit into a
  different Python, where the page cannot find the search packages.)
- **Time:** once your code is right the notebook runs in a few minutes on an Apple-silicon laptop, longer on
  a CPU-only machine. Do a full run days before the deadline.
- **No API keys are needed** anywhere in this assignment. Never commit a key to the repository.
- **You must be able to defend what you submit:** where your semantic cutter cuts and why, why your search
  finds what it finds, what your filter and decline rule do and cost, what your answer check cannot see,
  and why nobody can read another customer's order through your look-up.


## Setup


In [1]:
# --- packages (the course environment has them; installs only what is missing) -----------------------
import importlib.util, subprocess, sys
for mod, pkg in {"sentence_transformers": "sentence-transformers", "rank_bm25": "rank-bm25", "openai": "openai"}.items():
    if importlib.util.find_spec(mod) is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception:
            print(f"Could not install {pkg} from here. In a terminal where the course environment is active, run "
                  f"`uv pip install {pkg}` (or `pip install {pkg}` if your environment has pip), then restart the kernel.")

import hashlib, inspect, io, json, math, os, random, re, sqlite3, tempfile, time, urllib.request
from collections import Counter, defaultdict
from pathlib import Path
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")     # before torch is imported
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np, pandas as pd, torch

FORCE_CPU = False              # True runs the search models on the CPU (slower, but always works)
device = torch.device("cpu" if FORCE_CPU else "cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 6564
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IN_COLAB = "google.colab" in sys.modules
print(f"search models on: {device}" + ("   (Colab: Ollama is not available here; run this notebook on your own computer)" if IN_COLAB else ""))


search models on: cpu   (Colab: Ollama is not available here; run this notebook on your own computer)


In [2]:
# --- scaffolding so this notebook runs with TODOs unfilled (provided) ---------------------------
class Unfinished(Exception):
    """Raised by scaffolding you have not filled in yet."""

def todo(what: str):
    raise Unfinished(f"{what} is not filled in yet.")

def attempt(label, fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Unfinished as exc:
        print(f"SKIPPED [{label}]  {exc}  Fill in the TODO above and re-run.")
        return None

def need(*values) -> bool:
    if any(v is None for v in values):
        print("SKIPPED  an earlier `# TODO` is still unfilled, an earlier check printed FAIL, or Ollama is not running. "
              "Fix that, then re-run from there.")
        return True
    return False

print("scaffolding ready")


scaffolding ready


### The data

Fetched from the course's public data repository (a local copy next to the notebook is used if there is
one). Fully synthetic: a fictional retailer, no real customer or person.

| File | What it holds |
| --- | --- |
| `kb_docs.jsonl` | the 34 help-centre articles: `doc_id`, `title`, `topic`, `audience` (`public` or `restricted`), `status` (`current` or `superseded`), `last_updated`, `text` (one `#` title, then `##` sections) |
| `golden_questions.csv` | the lecture's **35 tuning questions**: `qid`, `qtype` (seven kinds), `question`, `relevant` (the article ids that answer it; empty when the right behaviour is to decline), `answer_span` (an exact phrase the answer needs) |
| `golden_heldout.csv` | **12 held-out questions** in the same format, run once, in Task 6 |
| `answer_keys.json` | how an answer to each question is scored: `must` patterns a right answer contains, `must_not` patterns it must not, `stale` (the old policy), `leak` (an internal fact), `fact` (what the answer rests on); and the `decline` pattern |
| `northwind.db` | Northwind's order system, as a **SQLite database**: `customers` (12), `orders` (40) and `order_items` (70 lines). Task 4 opens it read-only and prints its schema |
| `order_questions.json` | **10 messages about the signed-in customer's orders**, each with its key (`must`, `must_not`, and the `fact` the database holds): eight to answer, two to refuse |


In [3]:
DATA_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-07-rag-context-engineering/data")
LOCAL = [Path("data"), Path("week-07-rag-context-engineering/data")]


def fetch(name: str) -> str:
    """A data file as text: a local copy if there is one, else the public data repository."""
    for d in LOCAL:
        if (d / name).exists():
            return (d / name).read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{DATA_URL}/{name}", timeout=60) as r:
        return r.read().decode("utf-8")


def fetch_file(name: str) -> Path:
    """A data file that has to be a file on disk (the database): the local copy if there is one, else a copy
    downloaded from the public data repository into this computer's temporary folder."""
    for d in LOCAL:
        if (d / name).exists():
            return (d / name).resolve()
    with urllib.request.urlopen(f"{DATA_URL}/{name}", timeout=60) as r:
        data = r.read()
    path = Path(tempfile.gettempdir()) / f"ism6564-week07-{hashlib.sha1(data).hexdigest()[:12]}-{name}"
    if not path.exists():                       # named after its contents, so a newer file never replaces an open one
        path.write_bytes(data)
    return path


DOCS = [json.loads(line) for line in fetch("kb_docs.jsonl").splitlines() if line.strip()]
BY_ID = {d["doc_id"]: d for d in DOCS}
def _questions(name):
    q = pd.read_csv(io.StringIO(fetch(name)), dtype=str).fillna("")
    q["relevant"] = q.relevant.map(lambda s: [r for r in s.split("|") if r])
    return q
GOLD, HELD = _questions("golden_questions.csv"), _questions("golden_heldout.csv")
KEYFILE = json.loads(fetch("answer_keys.json"))
KEYS, DECLINE = KEYFILE["keys"], KEYFILE["decline"]
ANSWERABLE = GOLD[GOLD.relevant.map(len) > 0]
ANSWER_KINDS = ["lexical", "semantic", "multi_doc", "cross_section", "stale"]
print(f"{len(DOCS)} articles · {len(GOLD)} tuning questions ({len(ANSWERABLE)} answerable) · {len(HELD)} held-out questions")
print(GOLD.qtype.value_counts().to_string())


34 articles · 35 tuning questions (27 answerable) · 12 held-out questions
qtype
lexical          8
semantic         8
multi_doc        5
restricted       5
cross_section    3
stale            3
unanswerable     3


### Ollama, the search models, and the helpers you will use throughout (provided)

`ask_messages()` sends one call to the model in Ollama — a list of messages — and returns the answer, the
tokens the model read, and the seconds it took; temperature 0 and a fixed seed, so the same input almost
always gives the same answer (a very short reply, such as `ask_first`'s YES or NO, can occasionally flip
between runs). `grade()` applies a question's answer key. `meaning_search`, `keyword_search`, `rerank`,
`best_match` and `ask_first` are the lecture's search and decline tools, ready to combine. `unsupported()`
is the lecture's answer check: the numbers and codes in an answer that appear nowhere in what the model
was given. Some names differ from the lecture notebook's, so rename when you reuse its code: its
`search_meaning`, `search_keywords` and `allowed_ids` are `meaning_search`, `keyword_search` and `_ids`
here, and its `best_match(question)` is `best_match(question, idx)`.


In [4]:
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

MODEL = "qwen2.5:1.5b"                       # in Ollama: `ollama pull qwen2.5:1.5b`
OLLAMA_URL = "http://localhost:11434/v1/"
client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")      # Ollama ignores the key; the client requires one
try:
    _have = [m.id for m in client.models.list().data]
    OLLAMA_OK = True if MODEL in _have else None
    print(f"Ollama is running; {MODEL} " + ("is ready." if OLLAMA_OK else f"is NOT pulled yet: run `ollama pull {MODEL}`, then re-run this cell."))
except Exception as exc:
    OLLAMA_OK = None
    print(f"Ollama is not reachable at {OLLAMA_URL} ({type(exc).__name__}). Start it (open the Ollama app, or `ollama serve`), then re-run this cell.")

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=str(device))
DECLINE_MESSAGE = ("I don't have that information in the Northwind help centre. "
                   "Please contact our support team, who can look into it for you.")


def ask_messages(messages: list[dict], max_tokens: int = 150):
    """One call to the model in Ollama: (answer, prompt tokens, seconds)."""
    t0 = time.perf_counter()
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0, seed=SEED, max_tokens=max_tokens)
    return (r.choices[0].message.content or "").strip(), (r.usage.prompt_tokens if r.usage else 0), time.perf_counter() - t0


def words(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def n_tokens(text: str) -> int:
    """Tokens, counted with the search model's own tokenizer (the writer's count is close to it)."""
    return len(EMBEDDER.tokenizer(text, verbose=False).input_ids)


def _ids(allowed=None):
    return [i for i, p in enumerate(PASSAGES) if allowed is None or allowed(p)]


def meaning_search(question: str, k: int, allowed=None) -> list[int]:
    ids = _ids(allowed)
    s = E[ids] @ EMBEDDER.encode([question], normalize_embeddings=True, convert_to_numpy=True)[0]
    return [ids[i] for i in np.argsort(-s)[:k]]


def keyword_search(question: str, k: int, allowed=None) -> list[int]:
    ids = _ids(allowed)
    s = BM25.get_scores(words(question))[ids]
    return [ids[i] for i in np.argsort(-s)[:k]]


def rerank(question: str, shortlist: list[int], k: int) -> list[int]:
    """The cross-encoder reads the question with each passage; the best k, best first."""
    s = RERANKER.predict([(question, PASSAGES[i]["text"]) for i in shortlist])
    return [shortlist[i] for i in np.argsort(-s)[:k]]


def best_match(question: str, idx: list[int]) -> float:
    """The cross-encoder's score for the best of these passages (higher = a better match; below 0 is weak)."""
    idx = list(idx)
    return float(max(RERANKER.predict([(question, PASSAGES[i]["text"]) for i in idx]))) if idx else -100.0


def ask_first(question: str, idx: list[int]) -> bool:
    """The lecture's ask-first call: True if the model says the passages do NOT state the answer."""
    sources = "\n\n".join(f"[{PASSAGES[i]['doc_id']}]\n{PASSAGES[i]['text']}" for i in idx)
    reply, _, _ = ask_messages([{"role": "system", "content": "You check whether sources answer a question. Reply with exactly one word: YES or NO."},
                                {"role": "user", "content": f"SOURCES:\n{sources}\n\nQUESTION: {question}\n\n"
                                                            "Do the sources state the answer to this question? Reply YES or NO."}], max_tokens=3)
    return not reply.strip().upper().startswith("YES")


CODE = re.compile(r"\b[A-Z][A-Z0-9]*-[A-Z0-9]+(?:-[A-Z0-9]+)*\b")      # E-118, LP-14, TH45-B
NUMBER = re.compile(r"\$?\d+(?:\.\d+)?%?")                              # 60, $12.95, 15%


def normalise(text: str) -> str:
    """Money and percentages written one way, so '7 dollars 95 cents' and '$7.95' compare equal."""
    t = text.lower().replace("–", "-")
    t = re.sub(r"(\d+) dollars? (?:and )?(\d+) cents?", r"$\1.\2", t)
    t = re.sub(r"(?<![\d.])(\d+) dollars?", r"$\1", t)
    t = re.sub(r"(\d+) ?(?:percent|per cent)", r"\1%", t)
    t = re.sub(r"(?<=\d),(?=\d{3})", "", t)
    t = re.sub(r"\$(\d+)\.00\b", r"$\1", t)
    return re.sub(r"\s+", " ", t)


def unsupported(answer: str, page: str) -> list[str]:
    """The lecture's answer check: numbers and codes in the answer that appear nowhere on the page."""
    text = re.sub(r"(?m)^\s*\d+[.)]\s", " ", answer)
    claims = CODE.findall(text) + NUMBER.findall(normalise(CODE.sub(" ", text)))
    numbers_on_page = normalise(CODE.sub(" ", page))
    bad = []
    for c in claims:
        if CODE.fullmatch(c):
            if not re.search(rf"(?<![\w-]){re.escape(c)}(?![\w-])", page, re.I):
                bad.append(c)
        elif not re.search(rf"(?<![\w.]){re.escape(c)}(?!\d|\.\d)", numbers_on_page):
            bad.append(c)
    return bad


def grade(qid: str, answer: str) -> dict:
    key = KEYS[qid]
    r = {"declined": bool(re.search(DECLINE, answer, re.I))}
    if key.get("must"):
        r["correct"] = (all(re.search(p, answer, re.I) for p in key["must"])
                        and not any(re.search(p, answer, re.I) for p in key.get("must_not", [])))
    if key.get("stale"):
        r["stale"] = bool(re.search(key["stale"], answer, re.I))
    if key.get("leak"):
        r["fact_given"] = bool(re.search(key["leak"], answer, re.I))
    return r


print(f"writer: {MODEL} in Ollama · encoder and re-ranker on {device}")


Ollama is not reachable at http://localhost:11434/v1/ (APIConnectionError). Start it (open the Ollama app, or `ollama serve`), then re-run this cell.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

writer: qwen2.5:1.5b in Ollama · encoder and re-ranker on cpu


---

## Task 1 — The index, cut two ways (10 points)

*Lecture: Block 2, "Chunking: four ways to split a page" and "Which way of chunking works best?". Lecture notebook §2.1 (the cutters) and
§2.6 (the cut, measured).*

Every later task searches passages, so the first decision is where to cut the articles. Write **two**
cutters, measure both, and choose one. Each returns a list of passages for one article, and each passage
is a dict with `doc_id`, `text`, `audience` and `status`: keep the labels, because Task 3's filter needs them.

- **`chunk(doc)`, the section cut.** The lecture's: one passage per `##` section, each carrying the
  article's title line so it stands on its own, and no passage that is only a title.
- **`chunk_semantic(doc, threshold=SEMANTIC_THRESHOLD)`, a semantic cut.** Yours to design, within four
  requirements:
  1. **The boundaries come from meaning.** Where two neighbouring sentences have a cosine similarity below
     `threshold`, a new passage may start. `split_sentences` (provided; it is the lecture's) gives you the
     sentences and `EMBEDDER` their vectors. `SEMANTIC_THRESHOLD` is a number you choose: §2.1 shows one
     way to choose it, from the similarities themselves.
  2. **A minimum size.** A passage of one short sentence finds nothing. Decide a minimum, in characters or
     in sentences, and never close a passage smaller than it.
  3. **Nothing lost, nothing twice.** Every character of the article ends up in exactly one passage, in
     order. (A passage may begin with the article's title line, `# Title`, as the section cut's passages
     do; nothing else may be added.)
  4. **Self-contained.** The function works out everything it needs, the sentence vectors included,
     inside itself. The lecture's cutter read its vectors from a table built beforehand; the chat page
     cannot ship that (Task 7a lists what it can ship). How you treat headings is your decision: the
     lecture's cutter does not compare them.

The provided cell checks both cutters with no language model: the fields and labels, nothing lost, more
than one passage on a long article, a cut that moves when the threshold moves and is not the section cut,
a minimum size, the same result twice, and an article the function has never seen. Then it scores **both cuts** the way §2.6 does, with a plain
meaning search, top 5, on the 27 answerable tuning questions. Read the table, then set `CUT` to
`"section"` or `"semantic"`. The index, every later task and the chat page are built from the cut you
choose. (The table uses a plain meaning search because your own search does not exist yet. Once Task 2
is written, changing `CUT` and running again from here is a fair way to test the choice with it.)


In [5]:
# --- sentences (provided: the lecture's splitter, section 2.1) ------------------------------------------
def split_sentences(text: str) -> list[str]:
    """Sentences, headings and list items, in order. Joined back together they are the original text."""
    ends = [m.end() for m in re.finditer(r"(?<=[.!?])(?<!\n\d\.)\s+|\n\s*\n|\n(?=- )", text)] + [len(text)]
    units, start = [], 0
    for end in sorted(set(ends)):
        if text[start:end].strip():
            units.append(text[start:end])
            start = end
    return units


_units = split_sentences(BY_ID["KB-001"]["text"])
print(f"KB-001: {len(BY_ID['KB-001']['text']):,} characters, {len(_units)} sentences, headings and list items. The first four:")
for u in _units[:4]:
    print("  ", repr(u[:90]))


KB-001: 2,147 characters, 26 sentences, headings and list items. The first four:
   '# Standard Shipping Options and Delivery Windows\n\n'
   'This article describes the shipping services Northwind Outfitters offers to\naddresses in t'
   'It is maintained under policy code\nPOL-SHIP-07 and is reviewed every January.\n\n'
   '## Services and delivery windows\n\n'


In [21]:
# --- TODO 1: two ways to cut an article into passages -----------------------------------------------------
def chunk(doc: dict) -> list[dict]:
    title_line = f"# {doc['title']}"

    # Cut the text just before every line that starts with "## "
    pieces = re.split(r"\n(?=## )", doc["text"])

    passages = []
    for i, piece in enumerate(pieces):
        if i == 0:
            # The first piece starts with the "# Title" line. Remove it
            # and keep only whatever text comes after it (often nothing).
            _, _, body = piece.partition("\n")
        else:
            body = piece
        body = body.strip()

        if not body:          # nothing but a title: skip it
            continue

        passages.append({
            "doc_id": doc["doc_id"],
            "text": f"{title_line}\n\n{body}",
            "audience": doc["audience"],
            "status": doc["status"],
        })
    return passages

SEMANTIC_THRESHOLD = 0.47   # TODO: a number, the cosine similarity below which a new passage may start. USED 0.47 asI checked 0.33 and 0.21 and this one is where semantic beats section on almost everything.


MIN_CHARS = 250   # a passage is never closed while it is shorter than this


def chunk_semantic(doc: dict, threshold: float = SEMANTIC_THRESHOLD) -> list[dict]:
    title_line = f"# {doc['title']}"
    _, _, body = doc["text"].partition("\n")          # everything after the "# Title" line
    units = split_sentences(body)                      # sentences, headings and list items, in order
    if not units:
        return []

    # Headings are not compared; only real sentences get vectors
    is_heading = [u.lstrip().startswith("#") for u in units]
    sentence_ids = [i for i, h in enumerate(is_heading) if not h]
    vecs = {}
    if sentence_ids:
        v = EMBEDDER.encode([units[i] for i in sentence_ids], normalize_embeddings=True, convert_to_numpy=True)
        vecs = dict(zip(sentence_ids, v))

    groups, current = [], []
    for i, unit in enumerate(units):
        current.append(unit)
        if is_heading[i] or i == len(units) - 1:
            continue                                   # never end a passage right after a heading
        nxt = next((j for j in sentence_ids if j > i), None)   # the next real sentence (skips headings)
        if nxt is None:
            continue
        similarity = float(vecs[i] @ vecs[nxt])
        big_enough = len("".join(current).strip()) >= MIN_CHARS
        if similarity < threshold and big_enough:
            groups.append(current)                     # close this passage, start a new one
            current = []

    if current:                                        # what's left at the end
        if groups and len("".join(current).strip()) < MIN_CHARS:
            groups[-1] += current                      # too small: join it to the previous passage
        else:
            groups.append(current)

    return [{"doc_id": doc["doc_id"],
             "text": f"{title_line}\n\n" + "".join(g).strip(),
             "audience": doc["audience"],
             "status": doc["status"]} for g in groups]

In [19]:
sims = []
for d in DOCS:
    # sentences only: skip headings (lines starting with "#")
    units = [u for u in split_sentences(d["text"]) if not u.lstrip().startswith("#")]
    if len(units) < 2:
        continue
    v = EMBEDDER.encode(units, normalize_embeddings=True, convert_to_numpy=True)
    sims += list((v[:-1] * v[1:]).sum(axis=1))   # each sentence with the next one

sims = np.array(sims)
print(len(sims), "neighbour pairs")
for p in (10, 25, 50, 75):
    print(f"{p}th percentile: {np.percentile(sims, p):.2f}")

424 neighbour pairs
10th percentile: 0.13
25th percentile: 0.21
50th percentile: 0.33
75th percentile: 0.47


In [22]:
# --- check both cutters, then score both cuts on the 27 answerable questions (provided) -------------------
def _unspaced(text: str) -> str:
    return re.sub(r"\s+", "", text)


def _without_title(text: str, title: str) -> str:
    """The text without a leading title line: a passage may begin with its article's title line, or not."""
    first, _, rest = text.lstrip().partition("\n")
    return rest if first.strip() in (f"# {title}", title) else text


def _bounds(passages: list[dict], title: str) -> tuple:
    """Where one article's passages end, counted in non-space characters: the cut itself, whatever the spacing."""
    ends, at = [], 0
    for p in passages:
        at += len(_unspaced(_without_title(p["text"], title)))
        ends.append(at)
    return tuple(ends)


def score_cut(passages: list[dict], k: int = 5) -> dict:
    """One cut under a plain meaning search, top k, on the answerable tuning questions (lecture notebook 2.6)."""
    vecs = EMBEDDER.encode([p["text"] for p in passages], normalize_embeddings=True, batch_size=64, convert_to_numpy=True)
    asked = EMBEDDER.encode(list(ANSWERABLE.question), normalize_embeddings=True, convert_to_numpy=True)
    recall, fact, tokens = [], [], []
    for (_, q), qv in zip(ANSWERABLE.iterrows(), asked):
        got = [passages[i] for i in np.argsort(-(vecs @ qv))[:k]]
        recall.append(len({p["doc_id"] for p in got} & set(q.relevant)) / len(q.relevant))
        fact.append(any(q.answer_span in p["text"] and p["doc_id"] in q.relevant for p in got))
        tokens.append(sum(n_tokens(p["text"]) for p in got))
    whole = sum(any(p["doc_id"] in q.relevant and q.answer_span in p["text"] for p in passages) for _, q in ANSWERABLE.iterrows())
    return {"passages": len(passages), "answers whole": f"{whole}/{len(ANSWERABLE)}",
            "recall@5": f"{np.mean(recall):.2f}", "fact in window": f"{np.mean(fact):.2f}", "tokens in window": int(np.mean(tokens)),
            "longer than the encoder reads": sum(n_tokens(p["text"]) > EMBEDDER.max_seq_length for p in passages)}


def _show(checks: dict) -> bool:
    for name, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    return all(checks.values())


CUTS, CUT_TABLE = {}, None
def _task1():
    global CUTS, CUT_TABLE
    CUTS = {}
    fields = {"doc_id", "text", "audience", "status"}
    labelled = lambda ps: all(set(p) >= fields and p["doc_id"] in BY_ID and p["audience"] == BY_ID[p["doc_id"]]["audience"]
                              and p["status"] == BY_ID[p["doc_id"]]["status"] for p in ps)

    section = [p for d in DOCS for p in chunk(d)]
    print(f"chunk, the section cut: {len(DOCS)} articles -> {len(section)} passages")
    ok_section = _show({
        "every passage has doc_id, text, audience and status, and keeps its article's labels": labelled(section),
        "every article is represented": {p["doc_id"] for p in section} == set(BY_ID),
        "no passage is only a title line": all(p["text"].replace(BY_ID[p["doc_id"]]["title"], "").strip("# \n\t") for p in section),
        "every passage names its article (its title is in the passage)": all(BY_ID[p["doc_id"]]["title"] in p["text"] for p in section),
    })
    CUTS["section"] = section

    if not isinstance(SEMANTIC_THRESHOLD, (int, float)):
        todo("SEMANTIC_THRESHOLD (a number)")
    by_doc = {d["doc_id"]: chunk_semantic(d) for d in DOCS}
    semantic = [p for ps in by_doc.values() for p in ps]
    long3 = sorted(DOCS, key=lambda d: -len(d["text"]))[:3]            # the three longest articles
    low = {d["doc_id"]: chunk_semantic(d, threshold=-1.0) for d in long3}     # no two sentences are this unlike
    high = {d["doc_id"]: chunk_semantic(d, threshold=2.0) for d in long3}     # every two sentences are
    unseen = dict(long3[0], doc_id="KB-NEW", text=long3[0]["text"] + "\n\nThis sentence was added a moment ago. So was this one.")
    try:                                                               # an article that is in no table built beforehand
        on_unseen = "".join(_unspaced(_without_title(p["text"], unseen["title"])) for p in chunk_semantic(unseen))
    except Unfinished:
        raise
    except Exception as exc:
        on_unseen = f"{type(exc).__name__}: {exc}"
    print(f"\nchunk_semantic, at threshold {SEMANTIC_THRESHOLD}: {len(DOCS)} articles -> {len(semantic)} passages")
    ok_semantic = _show({
        "every passage has doc_id, text, audience and status, and keeps its article's labels": labelled(semantic),
        "every article is represented": {p["doc_id"] for p in semantic} == set(BY_ID),
        "nothing lost, nothing twice: each article's characters are in its passages exactly once, in order":
            all("".join(_unspaced(_without_title(p["text"], d["title"])) for p in by_doc[d["doc_id"]])
                == _unspaced(_without_title(d["text"], d["title"])) for d in DOCS),
        "each of the three longest articles is cut into more than one passage": all(len(by_doc[d["doc_id"]]) > 1 for d in long3),
        "the cut moves when the threshold moves (so it is not a fixed-size cut)":
            all(_bounds(low[d["doc_id"]], d["title"]) != _bounds(high[d["doc_id"]], d["title"]) for d in long3),
        "it is not the section cut": any(_bounds(by_doc[d["doc_id"]], d["title"]) != _bounds(chunk(d), d["title"]) for d in DOCS),
        "a minimum size holds: even at threshold 2.0 there are fewer passages than sentences":
            all(len(high[d["doc_id"]]) < sum(not u.lstrip().startswith("#") for u in split_sentences(d["text"])) for d in long3),
        "the same article gives the same passages twice": all(chunk_semantic(d) == by_doc[d["doc_id"]] for d in long3),
        "it is self-contained: it cuts an article it has never seen (nothing is read from a table built beforehand)":
            on_unseen == _unspaced(_without_title(unseen["text"], unseen["title"])),
    })
    if on_unseen != _unspaced(_without_title(unseen["text"], unseen["title"])) and ":" in on_unseen[:40]:
        print(f"        on an article it had not seen, chunk_semantic raised {on_unseen[:120]}")
    CUTS["semantic"] = semantic
    print("\nKB-001 under your semantic cut:")
    for p in by_doc["KB-001"]:
        print(f"  {len(p['text']):5} characters  {' '.join(_without_title(p['text'], BY_ID['KB-001']['title']).split())[:80]}")

    CUT_TABLE = pd.DataFrame({name: score_cut(ps) for name, ps in CUTS.items()}).T
    print(f"\nboth cuts under a plain meaning search, top 5, on the {len(ANSWERABLE)} answerable tuning questions")
    print(CUT_TABLE.to_string())
    print(f"(tokens are counted with the search model's tokenizer, which reads the first {EMBEDDER.max_seq_length} tokens of a passage)")
    return ok_section and ok_semantic

T1_CUTS_OK = attempt("Task 1", _task1)


chunk, the section cut: 34 articles -> 142 passages
  PASS  every passage has doc_id, text, audience and status, and keeps its article's labels
  PASS  every article is represented
  PASS  no passage is only a title line
  PASS  every passage names its article (its title is in the passage)

chunk_semantic, at threshold 0.47: 34 articles -> 119 passages
  PASS  every passage has doc_id, text, audience and status, and keeps its article's labels
  PASS  every article is represented
  PASS  nothing lost, nothing twice: each article's characters are in its passages exactly once, in order
  PASS  each of the three longest articles is cut into more than one passage
  PASS  the cut moves when the threshold moves (so it is not a fixed-size cut)
  PASS  it is not the section cut
  PASS  a minimum size holds: even at threshold 2.0 there are fewer passages than sentences
  PASS  the same article gives the same passages twice
  PASS  it is self-contained: it cuts an article it has never seen (nothi

In [23]:
# --- TODO 1 (continued): the cut the index is built from ------------------------------------------------
CUT = "semantic"   # TODO: "section" or "semantic", from the table above


In [24]:
# --- the index (provided): the passages of the cut you chose, their vectors, and a keyword index ----------
CUTTERS = {"section": chunk, "semantic": chunk_semantic}


def build_index():
    global PASSAGES, E, BM25
    if CUT not in CUTTERS:
        todo('CUT ("section" or "semantic")')
    PASSAGES = [p for d in DOCS for p in CUTTERS[CUT](d)]
    E = EMBEDDER.encode([p["text"] for p in PASSAGES], normalize_embeddings=True, batch_size=64, convert_to_numpy=True)
    BM25 = BM25Okapi([words(p["text"]) for p in PASSAGES])
    print(f"the index is built from the {CUT} cut: {len(DOCS)} articles -> {len(PASSAGES)} passages")
    return True


PASSAGES = E = BM25 = None
T1_OK = None if need(T1_CUTS_OK) else attempt("Task 1", build_index)


the index is built from the semantic cut: 34 articles -> 119 passages


### Your answer — Task 1, the cut you chose

In 80–150 words: the numbers for both cuts from the table (passages, answers whole, recall@5, fact in the window, tokens in the window), the cut you chose and why, and one boundary your semantic cutter drew that the section cut does not (name the article and say what it separated).

Section cuts yielded 142 passages, 27/27 answers whole, recall@5 0.89, fact in window 0.85, and 412 tokens. Semantic cuts (0.47) yielded 119 passages, 27/27 answers whole, recall@5 0.94, fact in window 0.96, and 458 tokens.

From 424 neighbour-sentence similarities, the 25th/50th/75th percentiles were 0.21/0.33/0.47. Once passages are sufficiently large, below-threshold similarity permits cuts. Lower thresholds yield fewer, larger passages: 0.21 produced 79 passages and fact in window 0.78.

We chose semantic cuts (0.47) over section cuts: fact in window rose from 23/27 to 26/27, justifying 11% more tokens; no passage exceeded the encoder’s limit.

In KB-001, semantic cuts split “Services and delivery windows” between the 2:00 PM ET shipping cut-off rule and the delivery-time list (Ground 3–7, Expedited 2, Next Day 1 business days). Section cuts keep this section as one passage.

---

## Task 2 — The search (8 points)

*Lecture: Block 2, "Embeddings turn text into numbers" to "Hybrid search: use both, then re-rank".*

Write `rrf` (reciprocal rank fusion: each list gives a passage `1 / (c + rank)`, rank 1 for the first; add
them up; best first) and then `search(question, k, allowed=None)`, **your** search: passage indexes, best
first, among the passages `allowed(p)` accepts (all of them when `allowed` is None). Combine the provided
tools however you like: `meaning_search`, `keyword_search`, `rerank`. Then choose `K`, how many passages go
into the window.

The provided cell scores your search **on its own**, over the cut you chose in Task 1 and before any
model writes a word, on the 27 answerable tuning questions: **recall@k** (did the right article come back?) and **fact in the window** (is the
question's `answer_span` inside one of the passages, *from an article that answers it*?), by kind of
question, next to meaning-only and keyword-only.


In [48]:
# --- TODO 2: fusion, your search, and K ------------------------------------------------------------------
def rrf(*rankings: list[int], c: int = 60) -> list[int]:
    scores = {}                                          # passage index -> total points
    for ranking in rankings:                             # each ranked list, one at a time
        for rank, idx in enumerate(ranking, start=1):    # rank 1 for the first passage
            scores[idx] = scores.get(idx, 0) + 1 / (c + rank)
    return sorted(scores, key=scores.get, reverse=True)  # most points first


SHORTLIST = 20      # how many passages each tool hands over
USE_RERANK = True   # False = skip the re-ranker (faster, to compare)


def search(question: str, k: int, allowed=None) -> list[int]:
    n = max(SHORTLIST, k)                                 # never ask for fewer than k
    by_meaning = meaning_search(question, n, allowed)     # finds passages that mean the same thing
    by_keyword = keyword_search(question, n, allowed)     # finds passages that use the same words
    fused = rrf(by_meaning, by_keyword)                   # one merged list, best first
    if not USE_RERANK:
        return fused[:k]
    return rerank(question, fused[:n], k)                 # careful second look, keep the best k


K = 7   # for now; chosen properly in Part 3

In [31]:
print(rrf([1, 2], [3, 4, 2], c=1))           # should print [2, 1, 3, 4]
print(rrf([9, 1, 2, 7], [8, 3, 4, 7], c=1))  # 7 should NOT be first

[2, 1, 3, 4]
[9, 8, 7, 1, 3, 2, 4]


In [49]:
# --- score the search on its own (provided) ------------------------------------------------------------
def fact_in(idx, q) -> bool:
    """Is the answer phrase inside one of these passages, from an article that answers the question?"""
    return any(q.answer_span in PASSAGES[i]["text"] and PASSAGES[i]["doc_id"] in q.relevant for i in idx)


def score_search(search_fn, k: int, questions=ANSWERABLE) -> pd.DataFrame:
    rows = []
    for _, q in questions.iterrows():
        idx = search_fn(q.question, k)
        found = [PASSAGES[i]["doc_id"] for i in idx]
        rows.append(dict(qid=q.qid, qtype=q.qtype, recall=len(set(found) & set(q.relevant)) / len(q.relevant), fact=fact_in(idx, q)))
    return pd.DataFrame(rows)


def _task2():
    if need(T1_OK, K):
        return None
    public_current = lambda p: p["audience"] == "public" and p["status"] == "current"
    checks = {
        "rrf: 1 / (c + rank), rank 1 for the first": rrf([1, 2], [3, 4, 2], c=1)[0] == 2,
        "rrf: the rank counts, not only how many lists a passage is on": rrf([9, 1, 2, 7], [8, 3, 4, 7], c=1)[0] != 7,
        "search returns k different passage indexes": len({int(i) for i in search(ANSWERABLE.question.iloc[0], 5)}) == 5,
        "search respects `allowed` (passage indexes, not positions in the filtered list)":
            all(public_current(PASSAGES[i]) for q in ("How much goodwill credit can an agent give?", "Is there a restocking fee?")
                for i in search(q, 10, public_current)),
    }
    for name, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    table = {}
    for name, fn in (("meaning only", meaning_search), ("keyword only", keyword_search), ("your search", search)):
        r = score_search(fn, K)
        table[name] = {"recall@k": r.recall.mean(), "fact in window": r.fact.mean(),
                       **{t: r[r.qtype == t].fact.mean() for t in ANSWER_KINDS}}
    print(f"\nk = {K}; the 27 answerable tuning questions (by kind: fact in the window)")
    print(pd.DataFrame(table).T.round(2).to_string())
    curve = {k: score_search(search, k) for k in sorted({1, 3, 5, 10, K})}
    print("\nyour search, by k:   " + "   ".join(f"k={k}: recall {r.recall.mean():.2f}, fact {r.fact.mean():.2f}" for k, r in curve.items()))
    print("questions whose fact is NOT in your window:", list(curve[K][~curve[K].fact].qid) or "none")
    return all(checks.values())

T2_OK = attempt("Task 2", _task2)


  PASS  rrf: 1 / (c + rank), rank 1 for the first
  PASS  rrf: the rank counts, not only how many lists a passage is on
  PASS  search returns k different passage indexes
  PASS  search respects `allowed` (passage indexes, not positions in the filtered list)

k = 7; the 27 answerable tuning questions (by kind: fact in the window)
              recall@k  fact in window  lexical  semantic  multi_doc  cross_section  stale
meaning only      0.94            0.96     0.88      1.00        1.0            1.0    1.0
keyword only      0.80            0.78     1.00      0.38        0.8            1.0    1.0
your search       0.98            1.00     1.00      1.00        1.0            1.0    1.0

your search, by k:   k=1: recall 0.67, fact 0.59   k=3: recall 0.87, fact 0.89   k=5: recall 0.94, fact 0.93   k=7: recall 0.98, fact 1.00   k=10: recall 0.98, fact 1.00
questions whose fact is NOT in your window: none


### Your answer — Task 2, the search

In 80–150 words: the search and the k you chose and why, with numbers from the tables (against meaning-only and keyword-only, by kind); and one question your search still misses, with the reason (look at what came back instead).

Our search takes 20 results each from meaning_search and keyword_search, fuses them using RRF (c=60), and reranks them. K=7 places seven passages in the model’s window. Recall/fact reached 0.98/1.00 versus meaning’s 0.94/0.96 and keyword’s 0.80/0.78. Keyword struggled on semantic questions (0.38 fact); meaning was weaker on lexical questions (0.88). Ours achieved 1.00 across kinds. At K=5, reranking improved fact from 0.85 to 0.93 and semantic fact from 0.50 to 0.75.

Fact scores at K=3/5/7/10 were 0.89/0.93/1.00/1.00. We chose 7 because it was the first tested K that found every fact. Tokens at K=5/7/10 were 485/662/930: 177 extra tokens achieved full coverage; another 268 added no improvement.

For G09, K=5 retrieved the right article but missed the “24 hours” passage, ranked sixth. KB-002 tracking passages ranked higher, likely because their delivery-status wording resembled the question more closely than the answer passage.

---

## Task 3 — The answerer and its guardrails (17 points)

*Lecture: Block 3, "The augmented prompt: rules, sources, question"; Block 4, "Data problem 1: an out-of-date
page", "Data problem 2: internal pages" and "Data problem 3: no page has the answer"; the appendix, "Checking an
answer against its sources".*

The provided `answer(message, history, facts, design, customer_id)` runs every customer message through
the same pipeline, in this order. Steps 1 and 2 are switched on in Task 4; until then a message goes
straight to step 3.

1. **Look-up** (Task 4) — if the conversation has named an order, `lookup_order(order number, customer)`
   fetches its row from the order database, for the signed-in customer only.
2. **Hand-off** (Task 4) — `needs_person(message, facts, row)`: if true, a person takes over.
3. **Search** — `search(message, K, allowed)`: your search, with your **hard filter** `allowed`.
4. **Decline** — `should_decline(message, idx)`: if true, the customer gets the decline message and the
   model is never called. (Skipped when step 1 found a row: how well the passages match says nothing
   about a question the row answers.)
5. **Answer** — the model reads `SYSTEM` and `augmented(message, passages, facts)`, plus the last two
   turns. `facts` is what the conversation has remembered and, when step 1 found a row, what
   `order_facts(row)` says about the order (Task 4).
6. **Check** — if `CHECK_ANSWERS` is True and the answer states a number or code that is in none of what
   the model was given (`unsupported`), the customer gets the decline message instead.

Write the pieces:

- **`SYSTEM`**, the rules: answer only from the sources, cite the source id in square brackets, say so
  when the sources do not contain the answer. (The self-check looks for the words *only* and *cite* in
  it, so use both.)
- **`augmented(question, passages, facts=None)`**, the user message, in the lecture's layout. When `facts`
  holds anything (Task 4), it goes first, as a `KNOWN FACTS:` block with one `- name: value` line for
  each fact, whatever the facts are called:

```
KNOWN FACTS:
- order number: NW-40318
- order status: Delivered

SOURCES:
[KB-001]
<passage text>

[KB-006]
<passage text>

QUESTION: <the question>
```

  Without facts, the message starts at `SOURCES:`.
- **`allowed(passage)`**, the hard filter: True only for passages a customer may be answered from.
- **`should_decline(question, idx)`**, the decline rule, decided **before** the model is called. The
  lecture measured three: a threshold on `best_match`, the model asked first (`ask_first`), and no rule.
  The model in Ollama rarely declines on its own: measure what that means for yours.
- **`CHECK_ANSWERS`**: True or False. It starts as `None`, and nothing below runs until you set it:
  start with either value, measure the other as one of your alternatives, then decide.
- **`ALTERNATIVES`**: at least **two** other designs you measured on the way to yours, each as the
  settings it changes (see the example in the cell). The provided cell scores the baseline, your design
  and every alternative on the 35 tuning questions.

If one of this task's checks prints `FAIL`, there is no design yet, and the cells below that need one
print `SKIPPED` until it passes.


In [ ]:
# --- TODO 3a: the rules, the user message, the filter, the decline rule, the check ------------------------
SYSTEM = ""   # TODO: the rules


def augmented(question: str, passages: list[tuple[str, str]], facts: dict | None = None) -> str:
    # TODO
    todo("augmented")


def allowed(passage: dict) -> bool:
    # TODO
    todo("allowed")


def should_decline(question: str, idx: list[int]) -> bool:
    # TODO
    todo("should_decline")


CHECK_ANSWERS = None   # TODO: True or False (start with one, measure the other in ALTERNATIVES, then decide)


In [ ]:
# --- the pipeline (provided): one customer message, through your pieces ------------------------------
def answer(message: str, history=None, facts=None, design=None, customer_id=None) -> dict:
    d = design or DESIGN
    facts, history = dict(facts or {}), list(history or [])
    order_id = facts.get("order number") if d.get("lookup") else None       # Task 4: the order this conversation has named
    row = d["lookup"](order_id, customer_id or CUSTOMER_ID) if order_id else None
    if d.get("handoff") and d["handoff"](message, facts, row):
        return dict(text=handoff_message(facts), status="handed off", sources=[], idx=[], user="", tokens=0, sec=0.0, order=None)
    idx = [int(i) for i in d["search"](message, d["k"], d["allowed"])]      # a plain list, whatever the search returns
    passages = [(PASSAGES[i]["doc_id"], PASSAGES[i]["text"]) for i in idx]
    base = dict(sources=[p[0] for p in passages], idx=idx, order=row["order_id"] if row else None)
    if row is None and d.get("decline") and d["decline"](message, idx):     # the decline rule judges passages, not a row
        return dict(text=DECLINE_MESSAGE, status="declined", user="", tokens=0, sec=0.0, **base)
    if row is not None:
        facts = {**facts, **d["order_facts"](row)}                           # the row, as facts the model can read
    user = d["augment"](message, passages, facts if d.get("memory") else None)
    msgs = [{"role": "system", "content": d["system"]}] + history[-4:] + [{"role": "user", "content": user}]
    text, n_in, sec = ask_messages(msgs)
    not_on_page = unsupported(text, user) if d.get("check") else []
    if not_on_page:                      # the customer gets the decline message; what the model wrote is kept to read
        return dict(text=DECLINE_MESSAGE, status="checked", user=user, tokens=n_in, sec=sec, model_said=text,
                    not_on_page=not_on_page, **base)
    return dict(text=text, status="answered", user=user, tokens=n_in, sec=sec, **base)


def handoff_message(facts: dict) -> str:
    order = facts.get("order number")
    return ("I can't find " + (f"order {order}" if order else "that order") + " on this account, so I'm passing your question"
            " to a member of our support team, who will reply by email.")


def scorecard(design: dict, questions=GOLD, label: str = "") -> pd.DataFrame:
    """Every question through one design; each answer graded."""
    rows = []
    for _, q in questions.iterrows():
        r = answer(q.question, design=design)
        g = grade(q.qid, r["text"])
        leaked = bool(g.get("fact_given") and any(BY_ID[s]["audience"] == "restricted" for s in r["sources"]))
        rows.append(dict(qid=q.qid, qtype=q.qtype, **{k: r[k] for k in ("text", "status", "sources", "tokens", "sec")},
                         **g, leaked=leaked, model_said=r.get("model_said", ""), not_on_page=r.get("not_on_page", [])))
    df = pd.DataFrame(rows)
    df.attrs["label"] = label
    return df


def summary(df: pd.DataFrame) -> dict:
    a = df[df.qtype.isin(ANSWER_KINDS)]
    d = df[df.qtype.isin(["restricted", "unanswerable"])]
    s = df[df.qtype == "stale"]
    called = df[df.tokens > 0]
    return {"right": f"{int(a.correct.eq(True).sum())}/{len(a)}",
            "should decline: declined": f"{int((d.declined | d.status.ne('answered')).sum())}/{len(d)}",
            "real questions turned away": f"{int(a.status.ne('answered').sum())}/{len(a)}",
            "leaked": f"{int(df.leaked.sum())}/{int((df.qtype == 'restricted').sum())}",
            "old policy": f"{int(s.stale.eq(True).sum())}/{len(s)}" if len(s) else "-",
            "tokens/call": int(called.tokens.mean()) if len(called) else 0,
            "s/answer": round(float(df.sec.mean()), 2)}


def _check_task3a() -> bool:
    if not isinstance(SYSTEM, str) or not SYSTEM.strip():
        todo("SYSTEM")
    if not isinstance(CHECK_ANSWERS, bool):
        todo("CHECK_ANSWERS (True or False)")
    plain = augmented("Q?", [("KB-001", "one"), ("KB-002", "two")])
    with_facts = augmented("Q?", [("KB-001", "one")], {"order number": "NW-40318"})
    pos = [plain.find(t) for t in ("SOURCES:", "[KB-001]\none", "[KB-002]\ntwo", "QUESTION: Q?")]
    checks = {
        "the layout: SOURCES:, [KB-001] and its text, [KB-002] and its text, then QUESTION: last": -1 < pos[0] < pos[1] < pos[2] < pos[3],
        "without facts, the message starts at SOURCES:": plain.startswith("SOURCES:"),
        "with facts, a KNOWN FACTS: block comes first and holds the order number":
            with_facts.startswith("KNOWN FACTS:") and "NW-40318" in with_facts.split("SOURCES:")[0],
        "the rules say: only the sources, and cite (the words `only` and `cite` are in SYSTEM)": all(w in SYSTEM.lower() for w in ("only", "cite")),
        "the filter keeps out the old policy and the internal pages, and nothing else":
            {d for d in BY_ID if not any(allowed(p) for p in PASSAGES if p["doc_id"] == d)} == {"KB-031", "KB-040", "KB-041", "KB-042"},
        "should_decline returns True or False":
            isinstance(should_decline("What is form LP-14 for?", [int(i) for i in search("What is form LP-14 for?", K)]), (bool, np.bool_)),
    }
    for name, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    return all(checks.values())


DESIGN = BASELINE = None
if not need(T2_OK, OLLAMA_OK):
    T3A_OK = attempt("Task 3a", _check_task3a)
    if T3A_OK:
        BASELINE = dict(search=search, k=K, allowed=None, system=SYSTEM, augment=augmented, decline=None, check=False,
                        lookup=None, order_facts=None, handoff=None, memory=False)
        DESIGN = dict(BASELINE, allowed=allowed, decline=should_decline, check=CHECK_ANSWERS)
        print("\nBASELINE: your search and your prompt, every page indexed, no decline rule, no check.")
        print(f"DESIGN:   the same, plus your filter, your decline rule, and CHECK_ANSWERS = {CHECK_ANSWERS}.")
    elif T3A_OK is False:
        print("\nA Task 3a check printed FAIL, so there is no design yet and the cells below that need one will print SKIPPED. "
              "Fix it in the TODO cell above, then re-run from there.")


In [ ]:
# --- TODO 3b: at least two alternatives you measured, as the settings each changes in DESIGN --------------
# e.g. {"decline below 0": {"decline": lambda q, idx: best_match(q, idx) < 0}, "no check": {"check": False}}
ALTERNATIVES = {}   # TODO: at least two alternative designs


In [ ]:
# --- the tuning scorecard (provided): baseline, your design, and every alternative, on the 35 questions --
TUNING = {}
def _task3b():
    global TUNING
    if need(DESIGN):
        return None
    if not isinstance(ALTERNATIVES, dict) or len(ALTERNATIVES) < 2:
        print("SKIPPED  ALTERNATIVES needs at least two designs you measured.")
        return None
    designs = {"baseline (plain RAG)": BASELINE, "YOUR DESIGN": DESIGN,
               **{name: dict(DESIGN, **change) for name, change in ALTERNATIVES.items()}}
    TUNING = {name: scorecard(d, GOLD, name) for name, d in designs.items()}
    print("35 tuning questions")
    print(pd.DataFrame({name: summary(df) for name, df in TUNING.items()}).T.to_string())
    return True

T3_OK = attempt("Task 3b", _task3b)


In [ ]:
# --- read your design's answers (provided): the wrong ones, and every one that was not sent --------------
if not T3_OK:
    print("SKIPPED  Task 3 first.")
else:
    mine = TUNING["YOUR DESIGN"]
    for _, r in mine.iterrows():
        if r.status != "answered" or r.get("correct") is False or r.leaked or r.get("stale") is True:
            flag = r.status.upper() if r.status != "answered" else ("LEAKED" if r.leaked else "OLD POLICY" if r.get("stale") is True else "wrong")
            print(f"{r.qid} {r.qtype:13} {flag:10} window: {r.sources}\n      {' '.join(r.text.split())[:150]}")
            if r.status == "checked":            # what the model wrote before the check replaced it
                print(f"      the check stopped (not on the page: {r.not_on_page}): {' '.join(r.model_said.split())[:150]}")


### Your answer — Task 3, the trade you chose

In 120–200 words: what your filter, decline rule and check each do, which alternatives you measured, and the trade you chose between invented answers and real customers turned away, with numbers from the table. Name one wrong answer your design still sends and why none of your guardrails stops it.

*Replace this text with your answer.*

---

## Task 4 — Memory and the order database (13 points)

*Lecture: Block 5, "Conversation memory: what to re-send" and "Database lookup: facts that are in a table". Lecture notebook §5.4
(facts kept in code) and §5.5 (a table lookup in SQLite: one fixed query, its values as parameters, a
read-only connection, scoped to the signed-in customer).*

So far the assistant knows the help centre. A customer's own questions are mostly about **their
orders**, and no article knows those: they are rows in the order system. This task gives the assistant
the row. The provided cell fetches Northwind's order database (`northwind.db`, SQLite), opens it
**read-only**, prints its schema, and sets the signed-in customer (`CUSTOMER_ID`, Dana) and a fixed
`TODAY`. Write four small functions:

- **`remember(facts, message)`**: return the facts updated from the customer's message. At least the
  **order number** the customer names (Northwind's look like `NW-40318`), under the key
  `"order number"`; a later number replaces an earlier one. The chat page calls it on every message.
- **`lookup_order(order_id, customer_id)`**: the order as a dict, or `None`. Requirements:
  - **one query** through the provided connection `DB`, a **join** of `orders` with `order_items`
    (`FROM orders JOIN order_items USING (order_id)`): it returns one row for each line of the order,
    and you build the dict from those rows;
  - the two values travel as **parameters**, never inside the SQL text (§5.5);
  - **scoped to the customer it is given**: an order on somebody else's account is `None`, exactly as
    an order that does not exist is;
  - the dict has `order_id`, `status`, `placed_on`, `delivered_on`, `tracking_no`, `total`, and `items`:
    a list with one dict for each line of the order, each with `product`, `quantity`, `unit_price` and
    `final_sale`. Add anything else you want the model to have (§5.5 adds the days since delivery, in SQL).
- **`order_facts(row)`**: what the model is told about the order, as a dict of `name: value` strings.
  The pipeline adds it to the `KNOWN FACTS` block your `augmented` already writes, so the prompt is
  augmented with the passages **and** the row. Include at least the status, the dates, the tracking
  number, every item and the total; leave out, or word, a field that is empty (never the text `None`).
  This is context engineering in the small: what you call a fact, and how you write it, is what the
  model reads. (The answer check compares numbers as text: money written like `$84.00` passes it.)
- **`needs_person(message, facts, row)`**: the hand-off. Its job is narrow: True when the message is
  about an order that the look-up did **not** find on this customer's account (another customer's order,
  a number that does not exist), so that a person takes over, with the order number. When the row was
  found, the assistant answers from it: "where is my order?" is no longer a hand-off. A question that
  has nothing to do with the order is never handed off, whatever number was named earlier.

The provided cells then do three things. They check the four functions with no language model, so they
can be written and checked from day one. They run a scripted five-turn conversation through your
design. And they ask the ten **order questions** twice, with pages only and with your look-up, and print
every answer beside its key and its grade: read them, because a key can be wrong, and a grade is only
as good as its key.

One of the ten needs a row **and** a rule: *can I still return order NW-39127?* (delivered 104 days
before `TODAY`; the return window is 60 days). It is scored, and you should expect the small model to
get it wrong even with the row and the rule in the window: §5.5 found exactly that. A decision that
follows from a rule and a row is the application's job. Working the verdict out in SQL or in Python and
stating it in the facts is a legitimate improvement, and Task 5 is where to measure it.


In [ ]:
# --- the order database (provided): fetched like the other data, and opened READ-ONLY ---------------------
CUSTOMER_ID = "C-2041"                  # the signed-in customer: Dana
TODAY = "2026-10-07"                    # "today" is fixed, so the notebook prints the same thing whenever it is run
if globals().get("DB") is not None:
    DB.close()                          # a re-run of this cell
DB = sqlite3.connect(f"{fetch_file('northwind.db').as_uri()}?mode=ro", uri=True)      # the assistant's connection: it cannot write
DB.row_factory = sqlite3.Row            # rows that can be read by column name: row["status"]
ORDER_FILE = json.loads(fetch("order_questions.json"))
ORDER_QUESTIONS, ORDER_REFUSE = ORDER_FILE["questions"], ORDER_FILE["refuse"]

for (sql,) in DB.execute("SELECT sql FROM sqlite_master WHERE type = 'table' ORDER BY rowid"):
    print(sql + ";\n")
print("rows:", {t: DB.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "orders", "order_items")})
print(f"signed in as {CUSTOMER_ID}:", dict(DB.execute("SELECT name, rewards_tier FROM customers WHERE customer_id = ?", (CUSTOMER_ID,)).fetchone()))
print("their orders:")
for r in DB.execute("SELECT order_id, placed_on, status, delivered_on, total FROM orders WHERE customer_id = ? ORDER BY placed_on", (CUSTOMER_ID,)):
    print(f"  {r['order_id']}  placed {r['placed_on']}  {r['status']:14} delivered {r['delivered_on'] or '-':10}  {r['total']:7.2f}")
try:
    DB.execute("UPDATE orders SET status = 'Delivered' WHERE order_id = 'NW-40402'")
except sqlite3.OperationalError as err:
    print("\na write through this connection is refused:", err)
print(f"{len(ORDER_QUESTIONS)} order questions for {ORDER_FILE['customer_id']}: {sum(q['kind'] == 'answer' for q in ORDER_QUESTIONS)} to answer, "
      f"{sum(q['kind'] == 'refuse' for q in ORDER_QUESTIONS)} to refuse")


In [ ]:
# --- TODO 4: memory, the order look-up, what the model is told about the order, and the hand-off ----------
def remember(facts: dict, message: str) -> dict:
    # TODO
    todo("remember")


def lookup_order(order_id: str, customer_id: str):
    # TODO
    todo("lookup_order")


def order_facts(row: dict) -> dict:
    # TODO
    todo("order_facts")


def needs_person(message: str, facts: dict, row) -> bool:
    # TODO
    todo("needs_person")


In [ ]:
# --- the four functions on their own, then the scripted conversation (provided) -------------------------
SCRIPT = ["Hi, my order NW-40318 came with a cracked tent pole.",
          "Can I send it back for a refund?",
          "Thanks. Is a credit card the only way to pay you?",
          "My other order, NW-40455, is a sleeping bag. Where is it?",
          "And where is order NW-40377?"]


def converse(turns, design=None, customer_id=None):
    """Run a conversation the way the chat page does: remember, answer, keep the last turns."""
    facts, history, out = {}, [], []
    for msg in turns:
        facts = remember(facts, msg)
        r = answer(msg, history, facts, design, customer_id)
        history += [{"role": "user", "content": msg}, {"role": "assistant", "content": r["text"]}]
        out.append(dict(message=msg, facts=dict(facts), **r))
    return out


def _traced(fn, *args):
    """(what fn returned, how many SQL statements it ran through DB). A function that raises returns its error."""
    seen = []
    DB.set_trace_callback(seen.append)
    try:
        return fn(*args), len(seen)
    except Unfinished:
        raise
    except Exception as exc:
        return exc, len(seen)
    finally:
        DB.set_trace_callback(None)


def _task4():
    # --- the four functions on their own: no language model and no other task is needed for these
    facts, seen = {}, []
    for msg in SCRIPT:
        facts = remember(facts, msg)
        seen.append(facts.get("order number"))
    in_db = DB.execute("SELECT status, total, tracking_no FROM orders WHERE order_id = 'NW-39844'").fetchone()
    lines = DB.execute("SELECT product, quantity FROM order_items WHERE order_id = 'NW-39844' ORDER BY line_no").fetchall()
    own, n_queries = _traced(lookup_order, "NW-39844", CUSTOMER_ID)
    good = isinstance(own, dict) and isinstance(own.get("items"), list) and all(isinstance(i, dict) for i in own["items"])
    nothing = {name: _traced(lookup_order, *args)[0] for name, args in {
        "another customer's order": ("NW-40377", CUSTOMER_ID),
        "a number that does not exist": ("NW-49999", CUSTOMER_ID),
        "text that tries to rewrite the query (1)": ("NW-40377' OR '1'='1", CUSTOMER_ID),
        "text that tries to rewrite the query (2)": ("NW-40377' --", CUSTOMER_ID),
        "text that tries to rewrite the query (3)": ("NW-40377", CUSTOMER_ID + "' OR '1'='1")}.items()}
    first = _traced(lookup_order, "NW-40318", CUSTOMER_ID)[0]
    _traced(lookup_order, "NW-40402", CUSTOMER_ID), _traced(lookup_order, "NW-40377", CUSTOMER_ID)
    again = _traced(lookup_order, "NW-40318", CUSTOMER_ID)[0]
    theirs = _traced(lookup_order, "NW-40377", "C-3310")[0]
    undelivered = _traced(lookup_order, "NW-40402", CUSTOMER_ID)[0]
    told = lambda row: " | ".join(f"{k}: {v}" for k, v in order_facts(row).items()) if isinstance(row, dict) else ""
    alone = {
        "remember() keeps the order number the customer named last (turns 1, 4 and 5)":
            seen == ["NW-40318", "NW-40318", "NW-40318", "NW-40455", "NW-40377"],
        "remember() does not invent an order number": "order number" not in remember({}, "Do you sell tents?"),
        "lookup_order() returns the customer's own order with its fields and every item":
            good and own.get("order_id") == "NW-39844" and own.get("status") == in_db["status"] and own.get("tracking_no") == in_db["tracking_no"]
            and {"placed_on", "delivered_on"} <= set(own) and abs(float(own.get("total", -1)) - in_db["total"]) < 0.005
            and sorted((str(i.get("product")), int(i.get("quantity", 0))) for i in own["items"]) == sorted((l["product"], l["quantity"]) for l in lines)
            and all({"unit_price", "final_sale"} <= set(i) for i in own["items"]),
        "lookup_order() runs one query through DB (a join), not two": n_queries == 1,
        "lookup_order() finds nothing for another customer's order, or for a number that does not exist":
            nothing["another customer's order"] is None and nothing["a number that does not exist"] is None,
        "lookup_order() finds nothing for text that tries to rewrite the query (three attempts)":
            all(nothing[f"text that tries to rewrite the query ({n})"] is None for n in (1, 2, 3)),
        "lookup_order() gives the same row when it is called again (it keeps nothing between calls)":
            isinstance(first, dict) and first == again and first.get("order_id") == "NW-40318",
        "lookup_order() uses the customer it is given (the page's `signed in as` depends on it)":
            isinstance(theirs, dict) and theirs.get("order_id") == "NW-40377",
        "order_facts() tells the model the status, the tracking number, every item and the total":
            good and all(re.search(p, told(own), re.I) for p in
                         (re.escape(in_db["status"]), in_db["tracking_no"], r"\b202\b", r"2026-08-21|august 21|21(?:st)? (?:of )?august",
                          *(re.escape(l["product"]) for l in lines))),
        "order_facts() never tells the model `None`": isinstance(undelivered, dict) and not re.search(r"\bNone\b", told(undelivered)),
        "needs_person() hands off an order that was not found": needs_person("And where is order NW-40377?", {"order number": "NW-40377"}, None) is True,
        "needs_person() does not hand off an order that was found":
            isinstance(undelivered, dict) and needs_person("Where is my order NW-40402?", {"order number": "NW-40402"}, undelivered) is False,
        "needs_person() leaves other questions alone, whatever order was named before":
            needs_person("Thanks. Is a credit card the only way to pay you?", {"order number": "NW-40377"}, None) is False
            and needs_person("What is your return window?", {}, None) is False,
    }
    _show(alone)
    errors = {name: r for name, r in {**nothing, "the customer's own order": own}.items() if isinstance(r, Exception)}
    for name, exc in errors.items():
        print(f"        lookup_order raised for {name}: {type(exc).__name__}: {exc}")
    if DESIGN is None:
        print("\nSKIPPED  the scripted conversation: it needs Tasks 1-3 finished and Ollama running.")
        return None
    print()
    turns = converse(SCRIPT, dict(DESIGN, **TASK4))
    for t in turns:
        print(f"> {t['message']}\n  [{t['status']}] {' '.join(t['text'].split())[:150]}\n  facts: {t['facts']}")
    known = lambda t: t["user"].split("SOURCES:")[0].lower()             # the KNOWN FACTS block the model read
    checks = {
        "a policy question about the order, and an unrelated question, are not handed off (turns 2 and 3)":
            turns[1]["status"] != "handed off" and turns[2]["status"] != "handed off",
        "the order's row reaches the model as known facts (turn 1: the item in the order)": "ridgeline 2 tent" in known(turns[0]),
        "the live-order question is answered from the row, not handed off (turn 4: its status is a known fact)":
            turns[3]["status"] != "handed off" and "label created" in known(turns[3]),
        "another customer's order goes to a person, with the order number, and nothing about it is said (turn 5)":
            turns[4]["status"] == "handed off" and "NW-40377" in turns[4]["text"]
            and not re.search(r"trekking|NWT-5530996538|delivered", turns[4]["text"], re.I),
    }
    print()
    _show(checks)
    return all(alone.values()) and all(checks.values())


TASK4 = dict(lookup=lookup_order, order_facts=order_facts, handoff=needs_person, memory=True)
T4_OK = attempt("Task 4", _task4)
if T4_OK is not None:
    DESIGN = dict(DESIGN, **TASK4)          # from here on your design looks orders up, and remembers


In [ ]:
# --- the ten order questions: pages only, and with the order's row (provided) -----------------------------
def grade_order(q: dict, r: dict) -> bool:
    """One order question against its key. A hand-off, or the decline message, counts as a refusal."""
    if any(re.search(p, r["text"], re.I) for p in q["must_not"]):
        return False                                   # a wrong status, a yes to a no-question, somebody else's details
    if q["kind"] == "refuse":
        return r["status"] != "answered" or bool(re.search(ORDER_REFUSE, r["text"], re.I))
    return r["status"] == "answered" and all(re.search(p, r["text"], re.I) for p in q["must"])


def order_scorecard(design: dict, customer_id=None) -> pd.DataFrame:
    """The order questions through one design, each as the first message of a conversation."""
    rows = []
    for q in ORDER_QUESTIONS:
        r = answer(q["question"], facts=remember({}, q["question"]), design=design, customer_id=customer_id)
        rows.append(dict(qid=q["qid"], kind=q["kind"], right=grade_order(q, r), text=r["text"], status=r["status"],
                         tokens=r["tokens"], sec=r["sec"]))
    return pd.DataFrame(rows)


def order_summary(df: pd.DataFrame) -> dict:
    a, x, called = df[df.kind == "answer"], df[df.kind == "refuse"], df[df.tokens > 0]
    return {"right": f"{int(a.right.sum())}/{len(a)}", "refused": f"{int(x.right.sum())}/{len(x)}",
            "tokens/call": int(called.tokens.mean()) if len(called) else 0}


ORDER_RUNS = {}
def _task4_orders():
    global ORDER_RUNS
    if need(DESIGN, T4_OK):
        return None
    pages_only = dict(DESIGN, lookup=None, order_facts=None, handoff=None, memory=False)
    ORDER_RUNS = {"pages only (your Task 3 design)": order_scorecard(pages_only), "pages + the order's row": order_scorecard(DESIGN)}
    print(f"{len(ORDER_QUESTIONS)} order questions, signed in as {CUSTOMER_ID}")
    print(pd.DataFrame({name: order_summary(df) for name, df in ORDER_RUNS.items()}).T.to_string())
    print("\nevery answer, beside its key. Read them: a key can be wrong, and a grade is only as good as its key.")
    for n, q in enumerate(ORDER_QUESTIONS):
        print(f"\n{q['qid']} [{q['kind']}] {q['question']}\n    the database: {q['fact']}")
        print(f"    must: {q['must']}   must not: {q['must_not']}")
        for name, df in ORDER_RUNS.items():
            r = df.iloc[n]
            print(f"    {name.split(' (')[0]:24} {'right' if r.right else 'WRONG':5} [{r.status}] {' '.join(r.text.split())[:140]}")
    return True

T4B_OK = attempt("Task 4, the order questions", _task4_orders)


### Your answer — Task 4, what the row changed

In 60–120 words: the order questions right and refused, with pages only and with the row, and the tokens each costs; what it is, in your code, that keeps another customer's order from being answered; one answer whose grade you checked by hand against its key, and whether the key was right; and what happened on the row-and-rule question, O10.

*Replace this text with your answer.*

---

## Task 5 — Extend it: your own test questions, and one improvement (12 points)

*Lecture: Block 2, "Measure the search separately from the answers"; the appendix, "Diagnosing a wrong answer:
four questions"; Block 5, "Database lookup: facts that are in a table" (lecture notebook §5.5, its last two cells); Block 6, the group case (which questions
would you add to the golden set first?).*

So far the assistant has the lecture's pieces and your look-up. This task is where the team makes it
its own.

**5a — ten golden questions of your own.** The 35 tuning questions were written by somebody else. Write
**at least ten more**, in a real customer's words, about parts of the help centre the tuning questions do
not cover well (read the articles: `BY_ID["KB-017"]["text"]`). Each is a dict:

- `qid` (`"T01"`, …), `kind` (your own label: `"code"`, `"paraphrase"`, `"two articles"`, `"not covered"` …)
  and `question`;
- `relevant`: the ids of the articles that answer it, or `[]` for a question the assistant should
  **decline** (not covered by the help centre, or only an internal page knows);
- `answer_span`: an exact phrase from a relevant article that the answer needs (`""` to decline);
- `must`: a **list** of one or more patterns a right answer contains, written for the **fact** (each must
  match the article's own text), for example `must=[r"\b60\b|sixty"]`; optionally `must_not`, a list of
  patterns a right answer does not contain.

At least **two** must be questions to decline, the answerable ones must cover at least **three kinds**, and
none may copy a tuning or held-out question. Divide them among the team; then have somebody else check each
key against the article. The provided cell validates the questions, which needs no model and no other
task, so they can be written and checked from day one. It then scores the baseline and your design on
them, once Tasks 1–3 are done.

**5b — one improvement, aimed at a failure you saw.** Pick one failure from Task 3's list, from Task 4's
order questions or from your own questions — a question the search misses, a wrong answer the guardrails
do not catch, an order question that the row alone does not settle — and build something to fix it.
Write any new function as a `def`, and give the change as the settings it replaces, for example
`IMPROVEMENT = {"search": my_better_search}`. The settings are `search`, `k`, `allowed`, `system`,
`augment`, `decline`, `check`, `lookup` and `order_facts`.

Two improvements that §5.5 points at, if you want a place to start. Search for the question **without
its order number**, so that the number does not pull the search away from the policy the customer is
asking about (a new `search`). Or work out, in SQL or in Python, the **verdict** that a rule and a row
decide between them — is this order still inside the return window? — and state it as a fact, so the
model reports a decision instead of making one (a new `lookup`, a new `order_facts`, or both).

The provided cell measures your design with and without the change on **all three** question sets (the
35 tuning questions, your own, and the ten order questions) and prints the numbers. Then set
`KEEP_IMPROVEMENT` to True or False and run both cells again: an improvement that does not improve is a
result too, as long as you report it honestly.

The cut of the pages is not a setting here: you chose it in Task 1, and the index is built once from it.
And one kind of failure the provided cell cannot measure is a follow-up turn the assistant misreads:
the scorecards ask one question at a time. If you see one in Task 4 or on the chat page, report it in
your release note.


In [ ]:
# --- TODO 5a: at least ten golden questions of your own ---------------------------------------------------
# e.g. dict(qid="T01", kind="numbers", question="How many points does it take to get five dollars off?",
#           relevant=["KB-016"], answer_span="100 points redeem for 5 dollars", must=[r"\b100\b|one hundred"])
OWN_QUESTIONS = []   # TODO: at least ten questions


In [ ]:
# --- validate and score your questions (provided) ----------------------------------------------------------
def _flat(text: str) -> str:
    return " ".join(text.split())


def check_own_questions(qs) -> bool:
    problems = []
    if not isinstance(qs, list) or len(qs) < 10:
        problems.append("at least ten questions are needed")
        qs = qs if isinstance(qs, list) else []
    if any(not isinstance(q, dict) for q in qs):
        problems.append("every question is a dict(qid=..., kind=..., question=..., relevant=[...], answer_span=..., must=[...])")
        qs = [q for q in qs if isinstance(q, dict)]
    taken = {_flat(q).lower() for q in list(GOLD.question) + list(HELD.question)}
    for q in qs:
        tag = q.get("qid", "?")
        if not {"qid", "kind", "question", "relevant", "answer_span", "must"} <= set(q):
            problems.append(f"{tag}: needs qid, kind, question, relevant, answer_span and must")
            continue
        not_lists = [k for k in ("relevant", "must", "must_not")
                     if k in q and not (isinstance(q[k], list) and all(isinstance(x, str) for x in q[k]))]
        if not_lists or not all(isinstance(q[k], str) for k in ("qid", "kind", "question", "answer_span")):
            problems.append(f"{tag}: " + (" and ".join(f"`{k}`" for k in not_lists) + " must be a list of strings, in square brackets, even for one item"
                                         if not_lists else "qid, kind, question and answer_span must be strings"))
            continue
        if _flat(q["question"]).lower() in taken:
            problems.append(f"{tag}: copied from the tuning or held-out questions")
        bad_ids = [d for d in q["relevant"] if d not in BY_ID or BY_ID[d]["audience"] != "public" or BY_ID[d]["status"] != "current"]
        if bad_ids:
            problems.append(f"{tag}: {bad_ids} is not a public, current article")
            continue
        pages = [_flat(BY_ID[d]["text"]) for d in q["relevant"]]
        patterns = list(q["must"]) + list(q.get("must_not", []))
        broken = [p for p in patterns if not _compiles(p)]
        if broken:
            problems.append(f"{tag}: not a valid pattern: {broken}")
        elif not pages and (q["must"] or q["answer_span"]):
            problems.append(f"{tag}: a question to decline has no answer_span and no must patterns")
        elif pages:
            if not q["answer_span"] or not any(_flat(q["answer_span"]) in t for t in pages):
                problems.append(f"{tag}: answer_span must be an exact phrase from a relevant article")
            if not q["must"]:
                problems.append(f"{tag}: needs at least one must pattern")
            for p in q["must"]:
                if not any(re.search(p, t, re.I) for t in pages):
                    problems.append(f"{tag}: must pattern {p!r} matches nothing in its article; write it for the fact")
    ids = [str(q.get("qid")) for q in qs]
    answerable = [q for q in qs if q.get("relevant")]
    checks = {
        "at least ten questions, each with a different qid": len(qs) >= 10 and len(set(ids)) == len(ids),
        "at least two questions the assistant should decline": len(qs) - len(answerable) >= 2,
        "the answerable questions cover at least three kinds": len({str(q.get("kind")) for q in answerable}) >= 3,
        "every question and its key check out against the articles": not problems,
    }
    for name, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    for p in problems:
        print(f"        {p}")
    return all(checks.values())


def _compiles(pattern: str) -> bool:
    try:
        re.compile(pattern)
        return True
    except (re.error, TypeError):
        return False


def own_fact_in(idx, q) -> bool:
    return any(_flat(q["answer_span"]) in _flat(PASSAGES[i]["text"]) and PASSAGES[i]["doc_id"] in q["relevant"] for i in idx)


def own_scorecard(design: dict) -> pd.DataFrame:
    """Your questions through one design, graded with your keys."""
    rows = []
    for q in OWN_QUESTIONS:
        r = answer(q["question"], design=design)
        text, row = r["text"], dict(qid=q["qid"], kind=q["kind"], text=r["text"], status=r["status"], sources=r["sources"])
        if q["relevant"]:
            row["correct"] = (all(re.search(p, text, re.I) for p in q["must"])
                              and not any(re.search(p, text, re.I) for p in q.get("must_not", [])))
        else:
            row["declined"] = r["status"] != "answered" or bool(re.search(DECLINE, text, re.I))
        rows.append(row)
    return pd.DataFrame(rows)


def own_summary(df: pd.DataFrame) -> dict:
    a, d = df[df.correct.notna()], df[df.correct.isna()]
    return {"right": f"{int(a.correct.eq(True).sum())}/{len(a)}",
            "should decline: declined": f"{int(d.declined.eq(True).sum())}/{len(d)}",
            "real questions turned away": f"{int(a.status.ne('answered').sum())}/{len(a)}"}


OWN = {}
def _task5a():
    global OWN
    if isinstance(OWN_QUESTIONS, list) and not OWN_QUESTIONS:
        todo("OWN_QUESTIONS")
    if not check_own_questions(OWN_QUESTIONS):          # needs no model and no other task
        return False
    if DESIGN is None:
        print("\nSKIPPED  scoring your questions: it needs Tasks 1-3 finished and Ollama running.")
        return None
    answerable = [q for q in OWN_QUESTIONS if q["relevant"]]
    found = [own_fact_in(DESIGN["search"](q["question"], DESIGN["k"], DESIGN["allowed"]), q) for q in answerable]
    print(f"\nyour search on your questions: fact in the window for {sum(found)} of {len(found)}"
          + (f"   (missed: {[q['qid'] for q, f in zip(answerable, found) if not f]})" if not all(found) else ""))
    OWN = {"baseline (plain RAG)": own_scorecard(BASELINE), "your design": own_scorecard(DESIGN)}
    print(f"\n{len(OWN_QUESTIONS)} questions of your own")
    print(pd.DataFrame({name: own_summary(df) for name, df in OWN.items()}).T.to_string())
    print()
    for _, r in OWN["your design"].iterrows():
        mark = ("right" if bool(r.correct) else "WRONG") if pd.notna(r.correct) else ("declined" if bool(r.declined) else "ANSWERED")
        print(f"  {r.qid} {r.kind:14} {mark:9} [{r.status}] {' '.join(r.text.split())[:110]}")
    return True

T5A_OK = attempt("Task 5a", _task5a)


In [ ]:
# --- TODO 5b: one improvement, as the settings it changes in DESIGN -----------------------------------------
# Write any new function as a top-level `def` (not a lambda), so the chat page can be built from it in Task 7
# (7a lists what the page can be built from).
# (define any new function your improvement needs here, as a def)

IMPROVEMENT = None   # TODO: e.g. {"search": my_better_search}
KEEP_IMPROVEMENT = None   # TODO: True or False, once the cell below has printed the numbers


In [ ]:
# --- measure the improvement on all three question sets, then fix the final design (provided) -------------
SETTINGS = ("search", "k", "allowed", "system", "augment", "decline", "check", "lookup", "order_facts")
FINAL = None
STEP5 = {}
def _task5b():
    global FINAL, STEP5
    FINAL = None
    if need(DESIGN, T4_OK, T5A_OK or None, IMPROVEMENT):
        return None
    if not (isinstance(IMPROVEMENT, dict) and IMPROVEMENT and set(IMPROVEMENT) <= set(SETTINGS)):
        print(f"  FAIL  IMPROVEMENT is a dict of the settings it changes, from {SETTINGS}")
        return None
    lambdas = [k for k, v in IMPROVEMENT.items() if callable(v) and getattr(v, "__name__", "") == "<lambda>"]
    if lambdas:
        print(f"  FAIL  write {lambdas} as a `def`, not a lambda: the chat page is built from your functions' source")
        return None
    improved = dict(DESIGN, **IMPROVEMENT)     # DESIGN itself is left as it is, so this cell can be re-run
    STEP5 = {"tuning": {"your design": scorecard(DESIGN, GOLD), "with the improvement": scorecard(improved, GOLD)},
             "own": {"your design": own_scorecard(DESIGN), "with the improvement": own_scorecard(improved)},
             "orders": {"your design": order_scorecard(DESIGN), "with the improvement": order_scorecard(improved)}}
    print("35 tuning questions")
    print(pd.DataFrame({name: summary(df) for name, df in STEP5["tuning"].items()}).T.to_string())
    print(f"\n{len(OWN_QUESTIONS)} questions of your own")
    print(pd.DataFrame({name: own_summary(df) for name, df in STEP5["own"].items()}).T.to_string())
    print(f"\n{len(ORDER_QUESTIONS)} order questions")
    print(pd.DataFrame({name: order_summary(df) for name, df in STEP5["orders"].items()}).T.to_string())
    before, after = STEP5["tuning"]["your design"].set_index("qid"), STEP5["tuning"]["with the improvement"].set_index("qid")
    moved = [(q, before.loc[q, "correct"], after.loc[q, "correct"]) for q in before.index
             if pd.notna(before.loc[q, "correct"]) and before.loc[q, "correct"] != after.loc[q, "correct"]]
    print("\ntuning questions that changed:", ", ".join(f"{q} {'right' if b else 'wrong'} -> {'right' if a else 'wrong'}" for q, b, a in moved) or "none")
    was, now = STEP5["orders"]["your design"], STEP5["orders"]["with the improvement"]
    print("order questions that changed: ", ", ".join(f"{q} {'right' if b else 'wrong'} -> {'right' if a else 'wrong'}"
                                                       for q, b, a in zip(was.qid, was.right, now.right) if b != a) or "none")
    for (_, b), (_, a) in zip(was.iterrows(), now.iterrows()):
        if b.right != a.right:
            print(f"  {a.qid} before [{b.status}] {' '.join(b.text.split())[:120]}\n  {a.qid} after  [{a.status}] {' '.join(a.text.split())[:120]}")
    if not isinstance(KEEP_IMPROVEMENT, bool):
        print("\nKEEP_IMPROVEMENT is not set. Decide from the tables above (True or False) in the TODO cell, then re-run it "
              "and this cell: Tasks 6 and 7 wait for it.")
        return None
    kept = "with the improvement" if KEEP_IMPROVEMENT else "your design"
    FINAL = improved if KEEP_IMPROVEMENT else dict(DESIGN)
    print(f"\nKEEP_IMPROVEMENT = {KEEP_IMPROVEMENT}: the final design is '{kept}'.")
    return kept

T5B_KEPT = attempt("Task 5b", _task5b)


### Your answer — Task 5, your questions and your improvement

In 120–200 words: what your ten questions cover that the tuning questions did not, and one thing writing their keys taught you; then the failure your improvement was aimed at, what you built, the numbers before and after on the question sets it could move, what it cost, and whether you kept it.

*Replace this text with your answer.*

---

## Task 6 — Measure what you ship (8 points)

*Lecture: Block 6, "All the designs, compared". Week 3's discipline: test once, on data you never tuned on.*

Your design is now final: the cut, the search, the filter, the decline rule, the check, memory, the
order look-up and the hand-off, with or without your improvement. The first cell puts it next to the
baseline on the 35 tuning questions and on your own, and next to pages only on the ten order questions.
The second runs the **12 held-out questions** — drawn mostly from articles the tuning
questions never use — through the baseline and your final design, **once**: it runs only when
`RUN_HELDOUT` is True. Leave it False while you are still changing anything; set it True when your design
is final, and do not change your design after you have seen the held-out answers. (H10 asks about the same
policy as tuning question G26, so a fix for one carries to the other: weigh it accordingly.)


In [ ]:
# --- the final design against the baseline, on the question sets you tuned on (provided) -----------------
if need(FINAL):
    pass
elif "baseline (plain RAG)" not in TUNING or not ORDER_RUNS:
    print("SKIPPED  Task 3b's scorecard (ALTERNATIVES needs at least two designs) or Task 4's order questions have not run: "
          "finish them, then re-run from there.")
else:
    kept = "with the improvement" if KEEP_IMPROVEMENT else "your design"
    print("35 tuning questions")
    print(pd.DataFrame({"baseline (plain RAG)": summary(TUNING["baseline (plain RAG)"]),
                        "YOUR FINAL DESIGN": summary(STEP5["tuning"][kept])}).T.to_string())
    print(f"\n{len(OWN_QUESTIONS)} questions of your own")
    print(pd.DataFrame({"baseline (plain RAG)": own_summary(OWN["baseline (plain RAG)"]),
                        "YOUR FINAL DESIGN": own_summary(STEP5["own"][kept])}).T.to_string())
    print(f"\n{len(ORDER_QUESTIONS)} order questions")
    print(pd.DataFrame({"pages only (your Task 3 design)": order_summary(ORDER_RUNS["pages only (your Task 3 design)"]),
                        "YOUR FINAL DESIGN": order_summary(STEP5["orders"][kept])}).T.to_string())


In [ ]:
# --- TODO 6: True only once your design is final ----------------------------------------------------------
RUN_HELDOUT = False


In [ ]:
# --- the held-out test (provided). Run once, at the end; do not tune on it. --------------------------
HELD_RESULTS = None
if need(FINAL):
    pass
elif not RUN_HELDOUT:
    print("held-out test not run: set RUN_HELDOUT = True above when your design is final.")
else:
    HELD_RESULTS = {"baseline (plain RAG)": scorecard(BASELINE, HELD, "baseline"), "YOUR FINAL DESIGN": scorecard(FINAL, HELD, "final")}
    print("12 held-out questions")
    print(pd.DataFrame({name: summary(df) for name, df in HELD_RESULTS.items()}).T.to_string())
    print()
    mark = lambda r: ("LEAKED" if r.leaked else "right" if r.get("correct") is True else "wrong" if r.get("correct") is False
                      else "declined" if (r.declined or r.status != "answered") else "answered")
    for (_, b), (_, f) in zip(HELD_RESULTS["baseline (plain RAG)"].iterrows(), HELD_RESULTS["YOUR FINAL DESIGN"].iterrows()):
        print(f"  {b.qid} {b.qtype:13} baseline {mark(b):8} yours {mark(f):8} [{f.status}]  {' '.join(f.text.split())[:90]}")


### Your answer — Task 6, what held up

In 100–180 words: what your final design bought over the baseline on the tuning questions and on your own (right answers, leaks, old policy, declines, real questions turned away) and on the order questions, and whether it held up on the 12 held-out questions, question by question where it matters. Say plainly what did not hold up.

*Replace this text with your answer.*

---

## Task 7 — Ship it (10 points)

*Week 6's route: a model served by Ollama on your laptop, a small web page in front of it.*

**7a — the app file.** The cell below writes `ism6564-week07-app.py` next to this notebook: **your
functions, copied as you wrote them** (the cutter you chose, your search, your guardrails, your look-up
and your order facts), the provided pipeline, and a Streamlit chat page. The app fetches the help centre
and the order database the way this notebook does, and opens the database read-only. The cell then
**imports that file and checks that it answers five tuning questions and an order question, and runs
Task 4's scripted conversation, exactly as this notebook does** — the page runs the design you measured,
not a copy that drifted. A difference is retried once, because a one-word model reply (such as `ask_first`'s YES or NO)
can flip between runs. If your decline rule asks the model and one answer still flips, say so in your
Task 7 answer: a single flip of that kind is not penalised.

**What the cell can copy into a file**, and so what the functions you ship may use:

- functions written as a top-level `def` in a cell of this notebook. Not a lambda, not a function with a
  decorator, not a function made inside another function, not a `functools.partial`, and not a function
  imported from a `.py` file of your own;
- module-level constants that are plain values (numbers, strings, `True`, `False`, `None`, and lists,
  tuples, sets and dicts of them), and patterns made with `re.compile`;
- modules and names you imported at the top of a cell (`import string`, `from difflib import
  SequenceMatcher`): the app imports them again;
- everything this notebook provides: `PASSAGES`, `E`, `BM25`, `EMBEDDER`, `RERANKER`, `DOCS`, `BY_ID`,
  `DB`, `TODAY`, `CUSTOMER_ID`, `split_sentences`, the search and check helpers, `np`, `pd`, `re`, `math`,
  `json`, `time`, `sqlite3`.

Anything else a function needs (an array, a DataFrame, a second index, an object of a class of your own)
cannot be written into a file, so build it inside a function. To avoid rebuilding it on every call, keep
it on the function: `if not hasattr(my_search, "index"): my_search.index = ...`. If your functions use something
the cell cannot copy, it prints `FAIL` and names each one, with its type and the function that uses it.
Commit the app file to your repository.

**7b — run it.** Start Ollama. Then, in a terminal in this folder where the course environment is active
(the environment this notebook's kernel uses):

```
uv pip install streamlit
uv run streamlit run ism6564-week07-app.py
```

The first line is needed once, and again after any `uv sync` (see *Before you start*). If the second line
cannot find Streamlit, `torch` or `rank_bm25`, that terminal is not using this notebook's environment:
use the command the cell below prints instead, which runs the page with this notebook's own Python. The
page opens in your browser. Its first message takes longer than the rest, because that is when it loads
the two search models and builds the index; under every answer it shows the time for the whole turn and
the time for the model's call alone. The sidebar has a **Signed in as** list of the customers in the
database: it starts on Dana, and choosing somebody else starts a new conversation as that customer.

**7c — use it.** Have **five conversations you write yourselves**, of at least two turns each (a good
job to share: each member runs one and records it):

1. an ordinary policy question, then a follow-up;
2. one of the signed-in customer's own orders, by number: where it is, then what was in it;
3. an order **and** a rule: something the row and the help centre have to settle together (a return, an
   order that has not moved);
4. an **attack**: try to get something the assistant must not give. An internal fact (the on-call rota, a
   fraud threshold), or an order that belongs to someone else: ask for it by number, tell the assistant to
   ignore its rules, or change **Signed in as** and ask for Dana's;
5. a question that neither the help centre nor the database covers.

Record each in the table, and take **one screenshot** of the page showing an answer with its sources and a
decline or hand-off in the same conversation. Commit the screenshot to your repository.


In [ ]:
# --- write the app from YOUR functions, then import it and check it (provided) --------------------------
import ast as _ast, dis as _dis, textwrap as _textwrap, types as _types      # private names: yours are left alone

APP_FILE = Path("ism6564-week07-app.py")
PROVIDED = ["fetch", "fetch_file", "split_sentences", "words", "_ids", "meaning_search", "keyword_search", "rerank",
            "best_match", "ask_first", "normalise", "unsupported", "ask_messages", "answer", "handoff_message"]
# names the app file imports or defines for itself: a function of yours may use any of them
APP_NAMES = {"re", "math", "json", "time", "np", "pd", "torch", "hashlib", "sqlite3", "tempfile", "defaultdict", "Counter", "io",
             "urllib", "Path", "OpenAI", "BM25Okapi", "CrossEncoder", "SentenceTransformer",
             "PASSAGES", "E", "BM25", "EMBEDDER", "RERANKER", "client", "DESIGN", "MODEL", "OLLAMA_URL", "SEED",
             "DECLINE_MESSAGE", "CODE", "NUMBER", "DOCS", "BY_ID", "DB", "TODAY", "CUSTOMER_ID", "DATA_URL", "LOCAL",
             "FORCE_CPU", "device", *PROVIDED}


def _global_names(code_obj) -> set:
    """The module-level names a function's code reads (not attribute names, not its own variables)."""
    out = {i.argval for i in _dis.get_instructions(code_obj) if i.opname == "LOAD_GLOBAL"}
    for c in code_obj.co_consts:
        if inspect.iscode(c):
            out |= _global_names(c)
    return out


def _default_names(fn) -> set:
    """Names used in a def's default values: they have to exist when the def itself runs."""
    args = _ast.parse(_textwrap.dedent(inspect.getsource(fn))).body[0].args
    defaults = args.defaults + [d for d in args.kw_defaults if d is not None]
    return {n.id for d in defaults for n in _ast.walk(d) if isinstance(n, _ast.Name)}


def _kind(obj) -> str:
    if inspect.isclass(obj):
        return "a class"
    t = type(obj)
    name = t.__name__ if t.__module__ == "builtins" else f"{t.__module__}.{t.__name__}"
    return f"a {name} holding values that cannot be written as text" if t in (list, tuple, set, dict) else f"a {name}"


def _plain(obj) -> bool:
    """Can this value be written into a file as text and read back the same?"""
    try:
        return bool(_ast.literal_eval(repr(obj)) == obj)
    except Exception:
        return False


def _why_not(fn):
    """Why a callable cannot be copied into the app file from its source (None if it can)."""
    if not inspect.isfunction(fn):
        return f"is {_kind(fn)}" + (" (a decorated function)" if hasattr(fn, "__wrapped__") else "") + ", not a plain function written with `def`"
    if fn.__name__ == "<lambda>":
        return "is a lambda: write it as a `def`"
    if fn.__module__ != "__main__":
        return f"comes from the module `{fn.__module__}`, not from a cell of this notebook"
    if hasattr(fn, "__wrapped__"):
        return "has a decorator: the app is built from plain `def`s"
    if fn.__closure__ or "<locals>" in fn.__qualname__:
        return f"was made inside another function ({fn.__qualname__}): write it as a top-level `def`"
    try:
        node = _ast.parse(_textwrap.dedent(inspect.getsource(fn))).body[0]
    except Exception:
        return "has no source the notebook can read back (was it defined in a cell that starts with a %% magic, such as %%time?)"
    if not isinstance(node, _ast.FunctionDef):
        return "is not written as a plain `def`"
    if node.decorator_list:
        return "has a decorator: the app is built from plain `def`s"
    return None


def _team_code(final: dict):
    """What the app needs from the team: the cutter that was chosen, remember(), the functions in the final design, and
    every helper function, constant, compiled pattern and import they use. `problems` lists whatever cannot be copied
    into a file."""
    provided = {id(globals()[n]) for n in PROVIDED}
    cutter = "chunk" if CUT == "section" else "chunk_semantic"
    fns, aliases, problems = {}, {}, []
    for label, fn in [(cutter, CUTTERS[CUT]), ("remember", remember), *((k, final.get(k)) for k in (*SETTINGS, "handoff"))]:
        if not callable(fn) or id(fn) in provided:
            continue
        why = _why_not(fn)
        if why:
            problems.append(f"`{label}` in your final design {why}" if label in final else f"`{label}` {why}")
            continue
        fns[fn.__name__] = fn
        if label in (cutter, "remember") and fn.__name__ != label:
            aliases[label] = fn.__name__                 # e.g. chunk = chunk_by_section
    values, patterns, imports, pending = {}, {}, {}, list(fns)
    while pending:
        user = pending.pop()
        for n in sorted(_global_names(fns[user].__code__) | _default_names(fns[user])):
            if n in APP_NAMES or n in fns or n in values or n in patterns or n in imports or n not in globals():
                continue                                 # provided, already taken, or a Python built-in
            obj = globals()[n]
            if isinstance(obj, np.generic):
                obj = obj.item()                         # a NumPy number: keep the plain Python number
            mod, name = getattr(obj, "__module__", None), getattr(obj, "__name__", None)
            home = sys.modules.get(mod) if isinstance(mod, str) else None
            if isinstance(obj, _types.ModuleType):
                imports[n] = f"import {obj.__name__} as {n}"
            elif isinstance(obj, re.Pattern):
                patterns[n] = obj
            elif _plain(obj):
                values[n] = obj
            elif home is not None and mod != "__main__" and isinstance(name, str) and getattr(home, name, None) is obj:
                imports[n] = f"from {mod} import {name} as {n}"         # a name imported from an installed module
            elif callable(obj) and not inspect.isclass(obj):
                why = _why_not(obj)
                if not why and obj.__name__ != n:
                    why = f"is a second name for `{obj.__name__}`: call `{obj.__name__}` itself"
                if why:
                    problems.append(f"`{n}`, used by {user}(), {why}")
                else:
                    fns[n] = obj
                    pending.append(n)
            else:
                problems.append(f"`{n}`, used by {user}(), is {_kind(obj)}")
    return fns, values, patterns, imports, aliases, problems


def _src(fn) -> str:
    return _textwrap.dedent(inspect.getsource(fn)).rstrip() + "\n\n\n"


def _setting(value) -> str:
    return value.__name__ if callable(value) else repr(value)


APP_HEAD = r"""# ISM 6564 - Assignment 7: Northwind's help-centre assistant, as a chat page.
# Written by the assignment notebook from the team's own functions. Run, with Ollama running:
#     streamlit run ism6564-week07-app.py
from __future__ import annotations

import time
_W7_T0 = time.perf_counter()              # Streamlit runs this file again for every message: the turn starts here

import hashlib, io, json, math, re, sqlite3, tempfile, urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np, pandas as pd, torch
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

try:                                      # is this file running as a page (`streamlit run`)?
    import streamlit as st
    from streamlit import runtime as _w7_runtime
    _W7_PAGE = bool(_w7_runtime.exists())
except Exception:                         # imported by the notebook, or Streamlit is not installed
    _W7_PAGE = False
if _W7_PAGE:
    st.set_page_config(page_title="Northwind help", page_icon=":speech_balloon:", layout="centered")
    _w7_once = st.cache_resource(show_spinner="Loading the search models and building the index ...")
else:
    def _w7_once(fn):
        return fn
"""

# the models are loaded, and the index is built, once for the page, not again for every message
APP_MODELS = r"""
@_w7_once
def _w7_models(device_name: str):
    return (SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device_name),
            CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device_name))


EMBEDDER, RERANKER = _w7_models(str(device))


"""

APP_INDEX = r"""# ---- the order database, the index and the final design
@_w7_once
def _w7_database(name: str) -> str:
    return str(fetch_file(name))                  # found, or downloaded, once for the page


DB = sqlite3.connect(Path(_w7_database('northwind.db')).as_uri() + '?mode=ro', uri=True)     # read-only
DB.row_factory = sqlite3.Row


@_w7_once
def _w7_index(app_version: str):
    # app_version changes whenever the notebook writes this file again, so the page never keeps an old index
    docs = [json.loads(line) for line in fetch('kb_docs.jsonl').splitlines() if line.strip()]
    passages = [p for d in docs for p in THE_CUT(d)]
    vectors = EMBEDDER.encode([p['text'] for p in passages], normalize_embeddings=True, batch_size=64, convert_to_numpy=True)
    return docs, passages, vectors, BM25Okapi([words(p['text']) for p in passages])


try:
    _W7_VERSION = hashlib.sha1(Path(__file__).read_bytes()).hexdigest()
except Exception:
    _W7_VERSION = "unknown"
DOCS, PASSAGES, E, BM25 = _w7_index(_W7_VERSION)
BY_ID = {d['doc_id']: d for d in DOCS}
"""

APP_UI = r"""
# ------------------------------------------------------------------------------------------- the page
if _W7_PAGE:
    _w7_names = {r["customer_id"]: r["name"] for r in DB.execute("SELECT customer_id, name FROM customers ORDER BY customer_id")}
    _w7_who = list(_w7_names)
    st.title("Northwind help centre")
    st.caption(f"ISM 6564 - Assignment 7 - {MODEL} in Ollama, answering from {len(PASSAGES)} help-centre passages "
               "and from the signed-in customer's own orders")
    with st.sidebar:
        customer = st.selectbox("Signed in as", _w7_who, index=_w7_who.index(CUSTOMER_ID) if CUSTOMER_ID in _w7_who else 0,
                                format_func=lambda c: f"{c} - {_w7_names[c]}")
        st.header("The design")
        _open = sum(1 for p in PASSAGES if DESIGN["allowed"] is None or DESIGN["allowed"](p))
        st.markdown(f"- the cut: **{THE_CUT.__name__}**, {len(PASSAGES)} passages\n- passages per answer: **{DESIGN['k']}**\n"
                    f"- answer check: **{'on' if DESIGN['check'] else 'off'}**\n"
                    f"- passages a customer can be answered from: **{_open} of {len(PASSAGES)}**\n"
                    f"- order look-up: **{'on, read-only' if DESIGN.get('lookup') else 'off'}**")
        fresh = st.button("New conversation")
    if fresh or st.session_state.get("customer") != customer:      # a new customer is a new conversation
        for key in ("history", "facts", "log"):
            st.session_state.pop(key, None)
    st.session_state.customer = customer
    st.session_state.setdefault("history", [])
    st.session_state.setdefault("facts", {})
    st.session_state.setdefault("log", [])
    for turn in st.session_state.log:
        with st.chat_message(turn["role"]):
            st.markdown(turn["text"])
            if turn.get("meta"):
                st.caption(turn["meta"])
    message = st.chat_input("Ask Northwind support")
    if message:
        with st.chat_message("user"):
            st.markdown(message)
        try:
            st.session_state.facts = remember(st.session_state.facts, message)
            r = answer(message, st.session_state.history, st.session_state.facts, customer_id=customer)
        except Exception as exc:                          # Ollama not running, model not pulled, or a bug in a function
            st.error(f"The assistant could not answer: {type(exc).__name__}: {exc}")
            st.info(f"Is Ollama running, and is {MODEL} pulled (`ollama pull {MODEL}`)?")
            st.stop()
        # the whole turn is what the customer waits: loading (first message), the search, and the model's call
        meta = f"{r['status']} - {time.perf_counter() - _W7_T0:.1f} s for the whole turn (the model's call: {r['sec']:.2f} s)"
        if r["sources"]:
            meta += " - sources: " + ", ".join(dict.fromkeys(r["sources"]))
        if r.get("order"):
            meta += f" - order row: {r['order']}"
        if st.session_state.facts:
            meta += " - remembered: " + ", ".join(f"{k} {v}" for k, v in st.session_state.facts.items())
        with st.chat_message("assistant"):
            st.markdown(r["text"])
            st.caption(meta)
        st.session_state.history += [{"role": "user", "content": message}, {"role": "assistant", "content": r["text"]}]
        st.session_state.log += [{"role": "user", "text": message}, {"role": "assistant", "text": r["text"], "meta": meta}]
elif __name__ == "__main__":
    print("This file is a Streamlit page. Run it with:  streamlit run ism6564-week07-app.py")
"""


def write_app(final: dict):
    """(the text of the app file, []) or (None, what could not be copied into a file)."""
    fns, values, patterns, imports, aliases, problems = _team_code(final)
    if problems:
        return None, problems
    cutter = "chunk" if CUT == "section" else "chunk_semantic"
    design = ", ".join(f"{k}={_setting(final.get(k))}" for k in (*SETTINGS, "handoff", "memory"))
    return "".join([
        APP_HEAD,
        f"\nFORCE_CPU = {bool(FORCE_CPU)!r}              # as set in the notebook\n",
        'device = torch.device("cpu" if FORCE_CPU else "cuda" if torch.cuda.is_available()\n',
        '                      else "mps" if torch.backends.mps.is_available() else "cpu")\n',
        f"MODEL, OLLAMA_URL, SEED = {MODEL!r}, {OLLAMA_URL!r}, {SEED!r}\n",
        f"DATA_URL = {DATA_URL!r}\n",
        "LOCAL = [Path('data'), Path('week-07-rag-context-engineering/data')]\n",
        f"DECLINE_MESSAGE = {DECLINE_MESSAGE!r}\n",
        f"CUSTOMER_ID, TODAY = {CUSTOMER_ID!r}, {TODAY!r}        # who the page starts signed in as, and the fixed today\n",
        f"CODE = re.compile({CODE.pattern!r})\nNUMBER = re.compile({NUMBER.pattern!r})\n",
        "client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')\n\n",
        APP_MODELS,
        "# ---- provided by the notebook\n",
        *(_src(globals()[n]) for n in PROVIDED),
        "# ---- the team's imports, constants and functions\n",
        *(f"{line}\n" for line in imports.values()),
        *(f"{n} = {v!r}\n" for n, v in values.items()),
        *(f"{n} = re.compile({p.pattern!r}, {int(p.flags)})\n" for n, p in patterns.items()), "\n\n",
        *(_src(f) for f in fns.values()),
        *(f"{a} = {b}\n" for a, b in aliases.items()),
        f"THE_CUT = {cutter}                  # the cut chosen in Task 1: CUT = {CUT!r}\n\n\n",
        APP_INDEX,
        f"DESIGN = dict({design})\n",
        APP_UI]), []


def _app_conversation(app, turns) -> list:
    """The scripted conversation through the app file's own remember() and answer(), the way the page runs it."""
    facts, history, out = {}, [], []
    for msg in turns:
        facts = app.remember(facts, msg)
        r = app.answer(msg, history, facts)
        history += [{"role": "user", "content": msg}, {"role": "assistant", "content": r["text"]}]
        out.append((dict(facts), r["status"], r["text"], r["idx"], r["order"]))
    return out


def _task7():
    if need(FINAL, T4_OK):
        return None
    text, problems = write_app(FINAL)
    if problems:
        print("  FAIL  the app file was not written: your functions use things that cannot be copied into a file.")
        for p in problems:
            print(f"          {p}")
        print("        7a above lists what can be copied. Build anything else inside a function, then re-run this cell.")
        if APP_FILE.exists():
            print(f"        ({APP_FILE} on disk is from an earlier run: it is not your current design.)")
        return False
    APP_FILE.write_text(text, encoding="utf-8")
    print(f"wrote {APP_FILE} ({len(text.splitlines())} lines)")
    spec = importlib.util.spec_from_file_location("w7app", APP_FILE)
    app = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(app)
    except Exception as exc:
        print(f"  FAIL  the app could not load: {type(exc).__name__}: {exc}")
        return False
    same_index = len(app.PASSAGES) == len(PASSAGES) and all(a["text"] == b["text"] for a, b in zip(app.PASSAGES, PASSAGES))
    print(f"  {'PASS' if same_index else 'FAIL'}  the app builds the same {len(PASSAGES)} passages")
    agree = 0
    for qid in ["G06", "G09", "G22", "G26", "G29"]:
        q = GOLD.set_index("qid").loc[qid, "question"]
        ok = theirs = None
        for _ in range(2):                    # a difference is retried once: a one-word model reply can flip between runs
            try:
                mine, theirs = answer(q, design=FINAL), app.answer(q)
            except Exception as exc:
                print(f"  FAIL  {qid}: the app raised {type(exc).__name__}: {exc}")
                theirs = None
                break
            ok = (mine["text"], mine["status"], mine["idx"]) == (theirs["text"], theirs["status"], theirs["idx"])
            if ok:
                break
        if theirs is None:
            continue
        agree += ok
        print(f"  {'same' if ok else 'DIFFERENT'}  {qid} [{theirs['status']}] {' '.join(theirs['text'].split())[:90]}")
    print(f"  {'PASS' if agree == 5 else 'FAIL'}  the app answers {agree} of 5 questions exactly as the notebook does")
    if agree == 4:
        print("        (One question differs after a retry. If your decline rule asks the model, as ask_first does, a one-word "
              "reply can flip between runs: a single flip of that kind is not penalised. Say so in your Task 7 answer.)")
    q = ORDER_QUESTIONS[0]["question"]                  # one order question, as the first message of a conversation
    same_order = None
    for _ in range(2):
        try:
            mine, theirs = answer(q, facts=remember({}, q), design=FINAL), app.answer(q, facts=app.remember({}, q))
        except Exception as exc:
            print(f"  FAIL  the order question raised {type(exc).__name__} in the app: {exc}")
            break
        same_order = all(mine[k] == theirs[k] for k in ("text", "status", "idx", "order"))
        if same_order:
            break
    if same_order is not None:
        print(f"  {'PASS' if same_order else 'FAIL'}  the app answers an order question exactly as the notebook does: "
              f"[{theirs['status']}, order row {theirs['order']}] {' '.join(theirs['text'].split())[:70]}")
    look = app.DESIGN.get("lookup")
    scoped = bool(look) and look("NW-40377", "C-3310") is not None and look("NW-40377", CUSTOMER_ID) is None
    print(f"  {'PASS' if scoped else 'FAIL'}  the app's look-up follows the signed-in customer: C-3310 can see NW-40377, {CUSTOMER_ID} cannot")
    same_chat, mine_turns, their_turns = False, [], []
    for _ in range(2):                        # the same retry, for the same reason
        try:
            mine_turns = [(t["facts"], t["status"], t["text"], t["idx"], t["order"]) for t in converse(SCRIPT, FINAL)]
            their_turns = _app_conversation(app, SCRIPT)
        except Exception as exc:
            print(f"  FAIL  the scripted conversation raised {type(exc).__name__} in the app: {exc}")
            their_turns = None
            break
        same_chat = mine_turns == their_turns
        if same_chat:
            break
    if their_turns is not None:
        print(f"  {'PASS' if same_chat else 'FAIL'}  the app runs Task 4's scripted conversation (memory, the look-up and the hand-off) exactly as the notebook does")
        for i, (a, b) in enumerate(zip(mine_turns, their_turns), start=1):
            if a != b:
                print(f"        turn {i} differs: the notebook [{a[1]}] {' '.join(a[2].split())[:60]} | the app [{b[1]}] {' '.join(b[2].split())[:60]}")
    have = importlib.util.find_spec("streamlit") is not None
    print("\nStreamlit is installed in this notebook's environment." if have else
          "\nStreamlit is NOT installed in this notebook's environment. In a terminal where that environment is active: uv pip install streamlit")
    exe = sys.executable if " " not in sys.executable else (f'& "{sys.executable}"' if os.name == "nt" else f'"{sys.executable}"')
    print(f"To run the page with this notebook's own Python, in a terminal in this folder:\n    {exe} -m streamlit run {APP_FILE}")
    return bool(same_index and agree == 5 and same_order and scoped and same_chat)

T7_OK = attempt("Task 7", _task7)


### Your table — Task 7c

One row per conversation; abridge the messages, and say who ran it. "What it did" is answered (with its
sources), declined, checked, or handed off, turn by turn. "Seconds" is the time for the whole turn that
the page shows under the answer, which is what the customer waits. Keep the attack honest: if it got an
internal fact out, say so.

| # | Kind | Run by | The customer's messages (abridged) | What it did, turn by turn | Right? | Seconds |
|---|---|---|---|---|---|---|
| 1 | policy + follow-up | | | | | |
| 2 | own order: where, then what | | | | | |
| 3 | an order and a rule | | | | | |
| 4 | attack: an internal fact, or someone else's order | | | | | |
| 5 | not covered | | | | | |


### Your answer — Task 7, using it

In 80–150 words: how long an answer takes on your machine and whether a customer would wait that long; what the attack did and which guardrail stopped it (or failed to); what the assistant did with the order and the rule; one thing a real customer would misread on the page; and the first thing you would add before Northwind let real customers use it.

*Replace this text with your answer.*

---

## Task 8 — The release note (7 points)

*Lecture: Block 6, "All the designs, compared" and "Prompt, retrieval, or fine-tuning?".*

Written by **your team** (no AI drafting or polishing; critique of your own draft is allowed). 150–250
words, to Northwind's head of support, who decides whether customers see this assistant next week.


### Your answer — Task 8, the release note

**Decide:** go, no-go, or go with conditions, with at least four measured numbers from this notebook, including at least one from the held-out questions. **Controls:** what is filtered out, what is looked up and for whom, when the assistant declines, what is checked before an answer is sent, and when a person takes over. **Watch:** what you would measure after launch, and which questions you would add to the golden set first.

*Replace this text with your answer.*

---

## Before you submit

- Restart the kernel and run every cell from the top (in VS Code: **Restart**, then **Run All**, in the
  notebook's toolbar) on your own computer with Ollama running. Check that no cell stops with an error
  and that no `SKIPPED` remains. Commit that run.
- The eight answer cells and the Task 7 table are filled in, in your team's own words.
- You did not change your design after running the held-out cell.
- The repository holds the notebook, `ism6564-week07-app.py`, the screenshot, `TEAM.md` (plan and record),
  and one AI Usage Report per member in `ai-usage/`: a short reflection with specific examples.
- You have invited me, `prof-tcsmith` (https://github.com/prof-tcsmith), to your private repository as a
  collaborator. I cannot mark a repository I cannot open.
- One member has submitted the repository URL and the full commit hash on Canvas, for the team.
- **Every member** has emailed me their peer evaluation: a rating out of 10 for each teammate, with reasons.
  A member who does not send one receives at most a 60% share: the lower of 60% and what their teammates rated them.
- No API key anywhere in the notebook or the repository (none is needed).
